In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
from time import sleep
import os

from scipy.spatial import ConvexHull
from shapely.geometry import Polygon as ShapelyPolygon
from matplotlib.patches import Polygon as MplPolygon
from shapely.ops import unary_union
from shapely.geometry import Point
from matplotlib.ticker import MultipleLocator


In [2]:
# --- Overall + per-category accuracy of the existing rule engine against
# your hand labels -- which categories it gets wrong most often, and
# specifically what it confuses them for. Useful on its own (tells you where
# to spend rule-tuning effort) and as a sanity check before trusting the ML
# model's own evaluation later.
#
# Two separate files: rule_predictions_path has the rule engine's output
# (game_id, gsis_play_id, nflId, motion_index, player_name, start_frame,
# end_frame, rule_based_category); manual_labels_path has your hand labels
# (same key columns, plus manual_label). Merge on the columns that actually
# identify a single motion -- game_id/gsis_play_id/nflId/motion_index --
# rather than assuming both files line up row-for-row.

manual_labels_path = 'results_motions/motion_labels_true.csv'
rule_predictions_path = 'results_motions/motion_labels_rule_based.csv'

merge_keys = ['game_id', 'gsis_play_id', 'nflId', 'motion_index']

rule_df = pd.read_csv(rule_predictions_path, dtype={'game_id': str, 'gsis_play_id': str})
manual_df = pd.read_csv(manual_labels_path, dtype={'game_id': str, 'gsis_play_id': str})

# Drop the descriptive columns duplicated across both files (player_name,
# start_frame, end_frame) from the manual side before merging, so the merge
# doesn't produce _x/_y suffixed duplicates of them.
manual_df = manual_df[merge_keys + ['manual_label']]

label_check_df = rule_df.merge(manual_df, on=merge_keys, how='outer', indicator=True)

unmatched_manual_only = label_check_df[label_check_df['_merge'] == 'right_only']
if len(unmatched_manual_only):
    print(f"WARNING: {len(unmatched_manual_only)} manual labels have no matching rule prediction "
          f"(key mismatch, or that motion isn't in {rule_predictions_path})")

label_check_df = label_check_df[label_check_df['_merge'] == 'both'].drop(columns=['_merge'])
label_check_df = label_check_df[
    label_check_df['manual_label'].notna() & (label_check_df['manual_label'] != '')
].copy()

label_check_df['correct'] = label_check_df['rule_based_category'] == label_check_df['manual_label']

total = len(label_check_df)
match = int(label_check_df['correct'].sum())
rate = match / total if total else 0
print(f"{match} out of {total} motions correctly captured ({rate:.1%})")

def _error_breakdown(df, group_col):
    g = df.groupby(group_col)['correct'].agg(['sum', 'count'])
    g = g.rename(columns={'sum': 'correct', 'count': 'total'})
    g['incorrect'] = g['total'] - g['correct']
    g['error_rate'] = g['incorrect'] / g['total']
    return g[['total', 'correct', 'incorrect', 'error_rate']].sort_values('error_rate', ascending=False)

print()
print("--- Error rate by TRUE (manual) label -- which real motion types the rules miss most ---")
print(_error_breakdown(label_check_df, 'manual_label').to_string())

print()
print("--- Error rate by RULE-PREDICTED label -- when the rules say X, how often is it wrong ---")
print(_error_breakdown(label_check_df, 'rule_based_category').to_string())

print()
print("--- Confusion matrix: rows = true (manual) label, columns = rule-predicted label ---")
confusion = pd.crosstab(label_check_df['manual_label'], label_check_df['rule_based_category'])
print(confusion.to_string())

1299 out of 1409 motions correctly captured (92.2%)

--- Error rate by TRUE (manual) label -- which real motion types the rules miss most ---
              total  correct  incorrect  error_rate
manual_label                                       
Return           77       60         17    0.220779
Hop              89       71         18    0.202247
Exit             95       81         14    0.147368
Orbit Loop        9        8          1    0.111111
Bolt             20       18          2    0.100000
Orbit            29       27          2    0.068966
Jet             308      288         20    0.064935
Across          397      373         24    0.060453
Short           266      256         10    0.037594
Out             119      117          2    0.016807

--- Error rate by RULE-PREDICTED label -- when the rules say X, how often is it wrong ---
                     total  correct  incorrect  error_rate
rule_based_category                                       
Orbit                   4

In [3]:
# --- Overall + per-category accuracy of the backset-formation rule engine
# against your hand labels -- same pattern as the motion accuracy check
# above, but per-play (no nflId/motion_index) and with backfield_class
# renamed on each side before merging, since both files use that same
# column name and would otherwise collide into _x/_y suffixes.
rule_predictions_path = 'results_backset_formations/backset_labels_rule_based.csv'
manual_labels_path = 'results_backset_formations/backset_labels_true.csv'
merge_keys = ['game_id', 'gsis_play_id']

rule_df = pd.read_csv(rule_predictions_path, dtype={'game_id': str, 'gsis_play_id': str})
manual_df = pd.read_csv(manual_labels_path, dtype={'game_id': str, 'gsis_play_id': str})

rule_df = rule_df.rename(columns={'backfield_class': 'rule_based_backfield_class'})
manual_df = manual_df[merge_keys + ['backfield_class']].rename(columns={'backfield_class': 'manual_label'})

label_check_df = rule_df.merge(manual_df, on=merge_keys, how='outer', indicator=True)

unmatched_manual_only = label_check_df[label_check_df['_merge'] == 'right_only']
if len(unmatched_manual_only):
    print(f"WARNING: {len(unmatched_manual_only)} manual labels have no matching rule prediction "
          f"(key mismatch, or that play isn't in {rule_predictions_path})")

label_check_df = label_check_df[label_check_df['_merge'] == 'both'].drop(columns=['_merge'])
label_check_df = label_check_df[
    label_check_df['manual_label'].notna() & (label_check_df['manual_label'] != '')
].copy()

label_check_df['correct'] = label_check_df['rule_based_backfield_class'] == label_check_df['manual_label']

total = len(label_check_df)
match = int(label_check_df['correct'].sum())
rate = match / total if total else 0
print(f"{match} out of {total} backset formations correctly captured ({rate:.1%})")

def _error_breakdown(df, group_col):
    g = df.groupby(group_col)['correct'].agg(['sum', 'count'])
    g = g.rename(columns={'sum': 'correct', 'count': 'total'})
    g['incorrect'] = g['total'] - g['correct']
    g['error_rate'] = g['incorrect'] / g['total']
    return g[['total', 'correct', 'incorrect', 'error_rate']].sort_values('error_rate', ascending=False)

print()
print("--- Error rate by TRUE (manual) label -- which real backset formations the rules miss most ---")
print(_error_breakdown(label_check_df, 'manual_label').to_string())

print()
print("--- Error rate by RULE-PREDICTED label -- when the rules say X, how often is it wrong ---")
print(_error_breakdown(label_check_df, 'rule_based_backfield_class').to_string())

print()
print("--- Confusion matrix: rows = true (manual) label, columns = rule-predicted label ---")
confusion = pd.crosstab(label_check_df['manual_label'], label_check_df['rule_based_backfield_class'])
print(confusion.to_string())

1682 out of 1773 backset formations correctly captured (94.9%)

--- Error rate by TRUE (manual) label -- which real backset formations the rules miss most ---
                      total  correct  incorrect  error_rate
manual_label                                               
Ace Weak                  3        0          3    1.000000
Ace Strong                5        0          5    1.000000
Split                     1        0          1    1.000000
Fullback Strong           1        0          1    1.000000
Pistol I Strong           6        2          4    0.666667
Shotgun Split            33       13         20    0.606061
I Weak                   16        8          8    0.500000
I Strong                 23       13         10    0.434783
Shotgun Split Strong     14       10          4    0.285714
Shotgun Split Weak       11        9          2    0.181818
I                        41       35          6    0.146341
3+ Backfield Set          8        7          1    0.125000
S

In [4]:
# --- Overall + per-category accuracy of the WR-formation rule engine
# against your hand labels -- same pattern, per-play, with wr_class
# renamed on each side before merging.
rule_predictions_path = 'results_wr_formations/wr_labels_rule_based.csv'
manual_labels_path = 'results_wr_formations/wr_labels_true.csv'
merge_keys = ['game_id', 'gsis_play_id']

rule_df = pd.read_csv(rule_predictions_path, dtype={'game_id': str, 'gsis_play_id': str})
manual_df = pd.read_csv(manual_labels_path, dtype={'game_id': str, 'gsis_play_id': str})

rule_df = rule_df.rename(columns={'wr_class': 'rule_based_wr_class'})
manual_df = manual_df[merge_keys + ['wr_class']].rename(columns={'wr_class': 'manual_label'})

label_check_df = rule_df.merge(manual_df, on=merge_keys, how='outer', indicator=True)

unmatched_manual_only = label_check_df[label_check_df['_merge'] == 'right_only']
if len(unmatched_manual_only):
    print(f"WARNING: {len(unmatched_manual_only)} manual labels have no matching rule prediction "
          f"(key mismatch, or that play isn't in {rule_predictions_path})")

label_check_df = label_check_df[label_check_df['_merge'] == 'both'].drop(columns=['_merge'])
label_check_df = label_check_df[
    label_check_df['manual_label'].notna() & (label_check_df['manual_label'] != '')
].copy()

label_check_df['correct'] = label_check_df['rule_based_wr_class'] == label_check_df['manual_label']

total = len(label_check_df)
match = int(label_check_df['correct'].sum())
rate = match / total if total else 0
print(f"{match} out of {total} WR formations correctly captured ({rate:.1%})")

def _error_breakdown(df, group_col):
    g = df.groupby(group_col)['correct'].agg(['sum', 'count'])
    g = g.rename(columns={'sum': 'correct', 'count': 'total'})
    g['incorrect'] = g['total'] - g['correct']
    g['error_rate'] = g['incorrect'] / g['total']
    return g[['total', 'correct', 'incorrect', 'error_rate']].sort_values('error_rate', ascending=False)

print()
print("--- Error rate by TRUE (manual) label -- which real WR formations the rules miss most ---")
print(_error_breakdown(label_check_df, 'manual_label').to_string())

print()
print("--- Error rate by RULE-PREDICTED label -- when the rules say X, how often is it wrong ---")
print(_error_breakdown(label_check_df, 'rule_based_wr_class').to_string())

print()
print("--- Confusion matrix: rows = true (manual) label, columns = rule-predicted label ---")
confusion = pd.crosstab(label_check_df['manual_label'], label_check_df['rule_based_wr_class'])
print(confusion.to_string())

1711 out of 1772 WR formations correctly captured (96.6%)

--- Error rate by TRUE (manual) label -- which real WR formations the rules miss most ---
                    total  correct  incorrect  error_rate
manual_label                                             
Closed - Closed        23        3         20    0.869565
Open - Tank Wing        8        3          5    0.625000
Tank Wing - Open        5        2          3    0.600000
Pro - Trey              2        1          1    0.500000
Slot - Closed          11        9          2    0.181818
Wing - Wing            13       11          2    0.153846
Quads - Open            9        8          1    0.111111
Closed - Trey          13       12          1    0.076923
Trips - Slot           28       26          2    0.071429
Trey - Open            69       65          4    0.057971
Slot - Open            19       18          1    0.052632
Trey - Closed          19       18          1    0.052632
Pro - Wing             23       22     

In [5]:
# raw_tracking_data = pd.read_csv('game_data/phi_dal_tracking_2025_1.csv')  # 400
# raw_tracking_data = pd.read_csv('game_data/kc_lac_tracking_2025_1.csv')   # 500
# raw_tracking_data = pd.read_csv('game_data/tb_atl_tracking_2025_1.csv')   # 700
# raw_tracking_data = pd.read_csv('game_data/cin_cle_tracking_2025_1.csv')  # 701
# raw_tracking_data = pd.read_csv('game_data/mia_ind_tracking_2025_1.csv')  # 702
# raw_tracking_data = pd.read_csv('game_data/car_jax_tracking_2025_1.csv')  # 703 (Travis Hunter: 1307, 2618, 2687)
# raw_tracking_data = pd.read_csv('game_data/lv_ne_tracking_2025_1.csv')    # 704
# raw_tracking_data = pd.read_csv('game_data/ari_no_tracking_2025_1.csv')   # 705
# raw_tracking_data = pd.read_csv('game_data/pit_nyj_tracking_2025_1.csv')  # 706
# raw_tracking_data = pd.read_csv('game_data/nyg_was_tracking_2025_1.csv')  # 707
# raw_tracking_data = pd.read_csv('game_data/ten_den_tracking_2025_1.csv')  # 708
# raw_tracking_data = pd.read_csv('game_data/sf_sea_tracking_2025_1.csv')   # 709
# raw_tracking_data = pd.read_csv('game_data/det_gb_tracking_2025_1.csv')   # 710
# raw_tracking_data = pd.read_csv('game_data/hou_lar_tracking_2025_1.csv')  # 711
# raw_tracking_data = pd.read_csv('game_data/bal_buf_tracking_2025_1.csv')  # 712
raw_tracking_data = pd.read_csv('game_data/min_chi_tracking_2025_1.csv')  # 800



# This file already has duplicate 'game_id'/'gsis_play_id' columns baked into
# its header row (looks like it was exported from an earlier run, before the
# rename below was fixed, then re-read). Collapse any duplicate-named columns
# down to one before anything else touches them -- otherwise selecting
# 'game_id' or 'gsis_play_id' anywhere below returns a DataFrame instead of a
# Series and breaks every merge/filter that follows.
def _dedupe_columns(df):
    dupe_names = df.columns[df.columns.duplicated()].unique()
    for name in dupe_names:
        same_name_cols = df.loc[:, df.columns == name]
        first_col = same_name_cols.iloc[:, 0].astype(str)
        disagreements = same_name_cols.apply(lambda col: col.astype(str) != first_col).any(axis=1)
        if disagreements.any():
            print(f"WARNING: duplicate '{name}' columns disagree in {disagreements.sum()} rows -- keeping the first")
    return df.loc[:, ~df.columns.duplicated()]

raw_tracking_data = _dedupe_columns(raw_tracking_data)

# CRITICAL: the underlying rows themselves are duplicated too -- every real
# tracking frame for a player is repeated several times over (11x observed on
# one play). This isn't just wasted memory: every frame-offset calculation in
# this notebook (_find_motion_end's post-snap cap, CROSSING_WINDOW_FRAMES,
# etc.) is a positional row-index offset, so with 11x duplication "10 frames"
# was actually only ~1 real frame past the snap the whole time -- no amount
# of raising those constants could have fixed that. Collapse to exactly one
# row per (game_id, gsis_play_id, nflId, frame_id) before anything else
# does frame-position math.
dupe_row_count = raw_tracking_data.duplicated(subset=['game_id', 'gsis_play_id', 'nflId', 'frame_id']).sum()
if dupe_row_count:
    print(f"WARNING: {dupe_row_count} duplicate (game_id, gsis_play_id, nflId, frame_id) rows found -- collapsing to one each")
raw_tracking_data = raw_tracking_data.drop_duplicates(
    subset=['game_id', 'gsis_play_id', 'nflId', 'frame_id']
).reset_index(drop=True)

# The API pull already tags tracking rows with 'game_id'/'gsis_play_id'
# (pull_tracking_data's own .assign(...)), while the play_info side still
# carries 'gameId'/'gsisPlayId' from the merge -- renaming the latter straight
# to 'game_id' would collide and produce two columns both named 'game_id', so
# consolidate instead of blindly renaming.
def _consolidate_id_column(df, snake_col, camel_col):
    if snake_col in df.columns and camel_col in df.columns:
        mismatch = df[snake_col].astype(str) != df[camel_col].astype(str)
        if mismatch.any():
            print(f"WARNING: {mismatch.sum()} rows have mismatched {snake_col} vs {camel_col}")
        return df.drop(columns=[camel_col])
    elif camel_col in df.columns:
        return df.rename(columns={camel_col: snake_col})
    return df

raw_tracking_data = _consolidate_id_column(raw_tracking_data, 'game_id', 'gameId')
raw_tracking_data = _consolidate_id_column(raw_tracking_data, 'gsis_play_id', 'gsisPlayId')
raw_tracking_data = raw_tracking_data.rename(columns={'positionGroup_x': 'position_group'})

raw_tracking_data = raw_tracking_data[raw_tracking_data['nflId'].notna()].copy()
raw_tracking_data['game_id'] = raw_tracking_data['game_id'].astype(str)
raw_tracking_data['gsis_play_id'] = raw_tracking_data['gsis_play_id'].astype(str)
raw_tracking_data['nflId'] = raw_tracking_data['nflId'].astype(float).astype(int)

# --- Ball position at snap, approximated by the center's position at the snap --
# this dataset has no pre-computed ball position, but the center lines up
# directly over the ball at the snap, so their (x, y) is effectively the same.
center_snap_positions = (
    raw_tracking_data[
        (raw_tracking_data['position_x'] == 'C') &
        (raw_tracking_data['event'] == 'ball_snap')
    ]
    .drop_duplicates(subset=['game_id', 'gsis_play_id'])
    [['game_id', 'gsis_play_id', 'x', 'y']]
    .rename(columns={'x': 'x_ball_at_snap', 'y': 'y_ball_at_snap'})
)

# Collapse any duplicate-named columns in this derived frame specifically
# (game_id/gsis_play_id showing up twice here means the merge below would
# fail with "column label is not unique") -- keep the first of each.
center_snap_positions = center_snap_positions.loc[:, ~center_snap_positions.columns.duplicated()]

raw_tracking_data = raw_tracking_data.merge(
    center_snap_positions, on=['game_id', 'gsis_play_id'], how='left'
)

missing_center = (
    raw_tracking_data.loc[raw_tracking_data['x_ball_at_snap'].isna(), ['game_id', 'gsis_play_id']]
    .drop_duplicates()
)
if len(missing_center):
    print(f"WARNING: {len(missing_center)} plays have no player tagged 'C' at the snap -- "
          f"falling back to the center-most OL player (by y-coordinate) as a stand-in")

    # Whoever's actually snapping the ball can be mislabeled (e.g. tagged as
    # a guard instead of center) -- the true center still always lines up
    # closest to the middle of the O-line by y-coordinate (field width),
    # regardless of their labeled position. Use that OL player's position
    # as a fallback for any play where the strict 'C' lookup came up empty.
    ol_snap_rows = raw_tracking_data[
        (raw_tracking_data['position_group'] == 'OL') &
        (raw_tracking_data['event'] == 'ball_snap')
    ].copy()

    ol_group_median_y = ol_snap_rows.groupby(['game_id', 'gsis_play_id'])['y'].transform('median')
    ol_snap_rows['dist_from_median_y'] = (ol_snap_rows['y'] - ol_group_median_y).abs()

    center_like_positions = (
        ol_snap_rows.sort_values('dist_from_median_y')
        .drop_duplicates(subset=['game_id', 'gsis_play_id'], keep='first')
        [['game_id', 'gsis_play_id', 'x', 'y']]
        .rename(columns={'x': 'x_ball_at_snap_fallback', 'y': 'y_ball_at_snap_fallback'})
    )

    raw_tracking_data = raw_tracking_data.merge(
        center_like_positions, on=['game_id', 'gsis_play_id'], how='left'
    )
    raw_tracking_data['x_ball_at_snap'] = raw_tracking_data['x_ball_at_snap'].fillna(raw_tracking_data['x_ball_at_snap_fallback'])
    raw_tracking_data['y_ball_at_snap'] = raw_tracking_data['y_ball_at_snap'].fillna(raw_tracking_data['y_ball_at_snap_fallback'])
    raw_tracking_data = raw_tracking_data.drop(columns=['x_ball_at_snap_fallback', 'y_ball_at_snap_fallback'])

    still_missing = (
        raw_tracking_data.loc[raw_tracking_data['x_ball_at_snap'].isna(), ['game_id', 'gsis_play_id']]
        .drop_duplicates()
    )
    if len(still_missing):
        print(f"WARNING: {len(still_missing)} plays still have no ball position -- no OL players found at the snap either")

# --- Formation-classification view: one row per player at the line_set frame ---
tracking_data = raw_tracking_data[raw_tracking_data['event'] == 'line_set'].copy()
snap_tracking_data = raw_tracking_data[raw_tracking_data['event'] == 'ball_snap'].copy()

tracking_data['x_diff_from_snap'] = tracking_data['x'] - tracking_data['xAtSnap']
tracking_data['y_diff_from_snap'] = tracking_data['y'] - tracking_data['yAtSnap']
tracking_data['x_player_ball_norm_at_line_set'] = round(tracking_data['x_ball_at_snap'] - tracking_data['x'])
tracking_data['y_player_ball_norm_at_line_set'] = round(tracking_data['y_ball_at_snap'] - tracking_data['y'])

play_keys = ['game_id', 'gsis_play_id']
skill_positions = ['RB', 'FB', 'WR', 'HB', 'TE', 'H-B']

formation_results = {}

KeyboardInterrupt: 

In [ ]:
def resolve_side(pos_count, neg_count):
    if pos_count > neg_count:
        return 'positive'
    elif neg_count > pos_count:
        return 'negative'
    else:
        return None

def resolve_tier2(pos_count, neg_count):
    if pos_count == 0 or neg_count == 0:
        return None
    if pos_count > neg_count:
        return 'positive'
    elif neg_count > pos_count:
        return 'negative'
    else:
        return None

def safe_round(val, decimals=0):
    if pd.isna(val):
        return np.nan
    res = round(val, decimals)
    return int(res) if decimals == 0 else res

def strength_suffix(strength):
    if strength == 'strong':
        return ' Strong'
    elif strength == 'balanced':
        return ''
    else:
        return ' Weak'

def motion_suffix(players):
    motion_players = []
    for player in players:
        if player['x_diff_from_snap'] == 0 and player['y_diff_from_snap'] == 0:
            continue
        if player['position_group'] in ('DL', 'DB', 'LB'):
            continue

        y_line = player['y_player_ball_norm_at_line_set']
        y_snap = player['y_player_ball_norm_at_snap']

        motion_desc = None
        if (y_snap > 0 and y_line < 0) or (y_snap < 0 and y_line > 0):
            motion_desc = 'across'
        else:
            if abs(y_snap) < abs(y_line):
                motion_desc = 'short'
            elif abs(y_snap) > abs(y_line):
                motion_desc = 'out'

        if motion_desc:
            motion_players.append(f"{player['position_group']} {motion_desc}")

    if not motion_players:
        return ''
    return ' with ' + ', '.join(motion_players) + ' motion'

def average_locations(players, backfield_depth_threshold, ol_left_edge, ol_right_edge):
    offense_x, offense_y = [], []
    skill_x, skill_y = [], []
    receiver_x, receiver_y = [], []
    backfield_x, backfield_y = [], []
    defense_x, defense_y = [], []
    secondary_x, secondary_y = [], []
    front_x, front_y = [], []

    for player in players:
        x, y = player['x_player_ball_norm_at_snap'], player['y_player_ball_norm_at_snap']
        pos_group = player['position_group']

        if pos_group in ('QB', 'OL', 'WR', 'RB', 'TE'):
            offense_x.append(x); offense_y.append(y)
        if pos_group in ('WR', 'RB', 'TE'):
            skill_x.append(x); skill_y.append(y)
            in_tackle_box_width = ol_left_edge <= x <= ol_right_edge
            in_backfield_depth = y >= backfield_depth_threshold
            if in_tackle_box_width and in_backfield_depth:
                backfield_x.append(x); backfield_y.append(y)
            else:
                receiver_x.append(x); receiver_y.append(y)
        if pos_group in ('DL', 'DB', 'LB'):
            defense_x.append(x); defense_y.append(y)
        if pos_group == 'DB':
            secondary_x.append(x); secondary_y.append(y)
        if pos_group in ('DL', 'LB'):
            front_x.append(x); front_y.append(y)

    def _avg(xs, ys):
        if not xs:
            return (None, None)
        return (round(np.mean(xs), 2), round(np.mean(ys), 2))

    return (_avg(offense_x, offense_y), _avg(skill_x, skill_y), _avg(receiver_x, receiver_y),
            _avg(backfield_x, backfield_y), _avg(defense_x, defense_y),
            _avg(secondary_x, secondary_y), _avg(front_x, front_y))

def _side_label(count, detached_count):
    if pd.isna(count):
        return 'Unknown'
    count = int(count)
    if count == 0:
        return 'Nix'
    detached_count = int(detached_count) if pd.notna(detached_count) else -1
    label_map = {
        (1, 1): 'Open', (1, 0): 'Closed',
        (2, 2): 'Slot', (2, 1): 'Pro', (2, 0): 'Wing',
        (3, 3): 'Trips', (3, 2): 'Trey', (3, 1): 'Toad Wing', (3, 0): 'Tank Wing',
        (4, 4): 'Quads', (4, 3): 'Quay', (4, 2): 'Quads Wing', (4, 1): 'Quake Wing', (4, 0): 'Quart Wing',
        (5, 5): 'Quint', (5, 4): 'Quarry', (5, 3): 'Quill Wing', (5, 2): 'Quibble Wing', (5, 1): 'Quench Wing', (5, 0): 'Quasi Wing',
    }
    return label_map.get((count, detached_count), 'Unknown')

def empty_backfield(qb_alignment):
    if qb_alignment in ('UNDER CENTER', 'UNDER_CENTER'):
        return 'Empty'
    elif qb_alignment == 'PISTOL':
        return 'Pistol Empty'
    elif qb_alignment == 'SHOTGUN':
        return 'Shotgun Empty'
    else:
        # FIX: previously fell through with no return (implicit None) for any
        # qb_alignment value other than the three above -- e.g. missing/NaN
        # data, or a value spelled differently than expected. That silently
        # broke `backfield_desc + motion_desc` downstream. Fall back to the
        # generic label, matching the pattern single_backfield/double_backfield
        # already use for their own unrecognized-qb_alignment case.
        return 'Empty'

def single_backfield(item):
    backfield_x = item['backfield_locations']['x_player_ball_norm_at_line_set'].iloc[0]
    backfield_y = item['backfield_locations']['y_player_ball_norm_at_line_set'].iloc[0]

    backfield_qb_dist_x = round(backfield_x - current_qb_x)
    backfield_strength = round(backfield_y, 1)

    if item['formation_strength'] == 'split':
        strength = 'balanced'
    elif backfield_strength > 0 and item['formation_strength'] == 'positive':
        strength = 'strong'
    elif backfield_strength < 0 and item['formation_strength'] == 'negative':
        strength = 'strong'
    elif item['formation_strength'] == 'none':
        strength = 'balanced'
    else:
        strength = 'weak'

    if item['qb_alignment'] == 'UNDER_CENTER':
        if abs(backfield_qb_dist_x) <= 1:
            return 'Ace FB' + strength_suffix(strength)
        elif backfield_qb_dist_x != 0:
            return 'Ace'
        else:
            return 'Ace'
    elif item['qb_alignment'] == 'PISTOL':
        return 'Pistol'
    elif item['qb_alignment'] == 'SHOTGUN':
        if abs(backfield_qb_dist_x) <= 1:
            return 'Shotgun' + strength_suffix(strength) + ' FB'
        else:
            return 'Shotgun' + strength_suffix(strength)
    else:
        return 'Single RB'

def double_backfield(item, current_qb_x, current_qb_y):
    sorted_rbs = item['backfield_locations'].sort_values('x_player_ball_norm_at_line_set')
    rb1_x, rb1_y = round(sorted_rbs.iloc[0]['x']), round(sorted_rbs.iloc[0]['y'])
    rb2_x, rb2_y = round(sorted_rbs.iloc[1]['x']), round(sorted_rbs.iloc[1]['y'])

    rb1_qb_dist_x = round(rb1_x - current_qb_x)
    rb1_qb_dist_y = round(rb1_y - current_qb_y)
    rb2_qb_dist_x = round(rb2_x - current_qb_x)
    rb2_qb_dist_y = round(rb2_y - current_qb_y)

    backfield_strength = round((rb1_y + rb2_y) / 2, 1)

    if item['formation_strength'] == 'split':
        strength = 'balanced'
    elif backfield_strength > 0 and item['formation_strength'] == 'positive':
        strength = 'strong'
    elif backfield_strength < 0 and item['formation_strength'] == 'negative':
        strength = 'strong'
    elif item['formation_strength'] == 'none':
        strength = 'balanced'
    else:
        strength = 'weak'

    if item['qb_alignment'] == 'UNDER_CENTER':
        if rb1_qb_dist_y == 0:
            return 'I' if rb2_qb_dist_y == 0 else 'I' + strength_suffix(strength)
        elif rb2_qb_dist_y == 0:
            return 'I' if rb1_qb_dist_y == 0 else 'I' + strength_suffix(strength)
        elif rb1_y == rb2_y and rb1_qb_dist_x != 0 and rb2_qb_dist_x != 0:
            return 'I Cheat' + strength_suffix(strength)
        elif rb1_x == rb2_x:
            if rb1_qb_dist_x == 0 or rb2_qb_dist_x == 0:
                return 'Double FB' + strength_suffix(strength) if (rb1_qb_dist_x == 0 and rb2_qb_dist_x == 0) else 'Split FB'
            else:
                if rb1_qb_dist_y == 0 or rb2_qb_dist_y == 0:
                    suffix = strength_suffix(strength).strip()
                    return suffix if suffix else 'Balanced'
                elif (rb1_qb_dist_x > 0 and rb2_qb_dist_x > 0) or (rb1_qb_dist_x < 0 and rb2_qb_dist_x < 0):
                    return 'Double' + strength_suffix(strength)
                else:
                    return 'Split'
        else:
            return 'Ace I FB' + strength_suffix(strength) if rb2_qb_dist_x == 0 else 'Split' + strength_suffix(strength)

    elif item['qb_alignment'] == 'PISTOL':
        if rb1_qb_dist_x > 1:
            if rb2_qb_dist_x > 1:
                return 'Pistol' + strength_suffix(strength)
            elif rb2_qb_dist_x == 0:
                return 'Pistol I' + strength_suffix(strength)
            else:
                return 'Pistol I'
        elif rb2_qb_dist_x > 1:
            if rb1_qb_dist_x > 1:
                return 'Pistol' + strength_suffix(strength)
            elif rb1_qb_dist_x == 0:
                return 'Pistol I' + strength_suffix(strength)
            else:
                return 'Pistol I'
        else:
            if item['formation_strength'] == 'split':
                return 'Pistol I'
            elif rb1_qb_dist_y == 0:
                if rb2_qb_dist_x > 0 and item['formation_strength'] == 'positive':
                    return 'Pistol I Strong'
                elif rb2_qb_dist_x < 0 and item['formation_strength'] == 'negative':
                    return 'Pistol I Strong'
                else:
                    return 'Pistol I Weak'
            elif rb2_qb_dist_y == 0:
                if rb1_qb_dist_x > 0 and item['formation_strength'] == 'positive':
                    return 'Pistol I Strong'
                elif rb1_qb_dist_x < 0 and item['formation_strength'] == 'negative':
                    return 'Pistol I Strong'
                else:
                    return 'Pistol I Weak'
            else:
                return 'Pistol I'

    elif item['qb_alignment'] == 'SHOTGUN':
        if rb1_x == rb2_x:
            if rb1_qb_dist_x == 0:
                if (rb1_qb_dist_x > 0 and rb2_qb_dist_x > 0) or (rb1_qb_dist_x < 0 and rb2_qb_dist_x < 0):
                    return 'Shotgun Double' + strength_suffix(strength)
                else:
                    return 'Shotgun Split'
            elif rb1_x < current_qb_x:
                if (rb1_qb_dist_x > 0 and rb2_qb_dist_x > 0) or (rb1_qb_dist_x < 0 and rb2_qb_dist_x < 0):
                    return 'Shotgun Double FB' + strength_suffix(strength)
                else:
                    return 'Shotgun Split FB'
            else:
                return 'Shotgun Double' + strength_suffix(strength)
        else:
            return 'Shotgun I' + strength_suffix(strength) if rb1_y == rb2_y else 'Shotgun Split' + strength_suffix(strength)
    else:
        return 'Double RB'

In [ ]:
STOP_SPEED_THRESH = 1.0        # yd/s -- below this counts as "stopped"
STOP_MIN_FRAMES = 3            # consecutive frames below threshold to count as stopped
 
ROUTE_ANGLE_THRESH = 3.0       # deg/frame -- below this counts as "heading stabilized"
ROUTE_STABLE_FRAMES = 5        # consecutive stable frames to count as "running a route"
 
RETURN_ANGLE_THRESH = 150.0    # deg -- net heading reversal close to 180 counts as a return
RETURN_DIST_THRESH = 3.0       # yards -- net displacement from start must be under this
 
JET_SPEED_THRESH = 5         # yd/s -- average speed during crossing to count as jet
 
MIN_MOTION_FRAMES = 5          # skip segments shorter than this (noise guard)

RETURN_TRAVEL_DIST = 4

TACKLE_BOX_OFFSET = 3.5

BACKFIELD_DEPTH_THRESHOLD = 1.0
MIN_STARTING_OFFSET_FOR_CROSSING = 2.0
RETURN_ORBIT_TURN_THRESH = 250 
QB_Y_ALIGNMENT_THRESHOLD = 2.0
IS_BEHIND_QB_THRESHOLD = 0

CROSSING_WINDOW_FRAMES = 30     # frames past the snap the crossing-geometry
                                # checks (made_it_to_tackle_box, ever_crosses_ball)
                                # are allowed to look through -- wider than the
                                # motion-end cap (10 frames) since a fast crossing
                                # motion can still be mid-crossing right at the
                                # snap, but bounded so an unrelated route crossing
                                # later in the play doesn't get mistaken for it

SUSTAINED_STOP_FRAMES = 20      # frames (~2 sec) of sustained low speed that
                                # counts as the motion having genuinely concluded
                                # -- the crossing-geometry window is capped here if
                                # it happens, so a post-snap route that breaks to the
                                # other side of the ball isn't mistaken for the motion
                                # itself having crossed

BRIEF_STOP_FRAMES = 10          # frames (1 second) below STOP_SPEED_THRESH
                                # anywhere during a crossing motion -- a real
                                # jet/orbit sprint is continuous, so a full second
                                # of stopping partway through means it isn't one,
                                # regardless of the peak speed reached; forces
                                # the classification down to 'Across' instead

MOTION_MIN_SPEED = 1.5          # yd/s -- a player must hit at least this speed
                                # sometime between the 'man_in_motion' tag and the
                                # snap to count as actually being the one in motion
                                # (that event tag is frame-level, not player-level,
                                # so every player's row gets it -- this is what tells
                                # a genuinely-motioning player apart from everyone else)
PLAY_LIVE_EVENTS = {'handoff', 'pass_forward', 'lateral'}  # once one of these
                                # fires, the play is genuinely underway -- the
                                # player is executing their assignment (running
                                # with the ball, etc.), not still positioning
                                # pre-snap. Both the motion-end and crossing-
                                # window searches are capped at the first such
                                # event so a designed run/pass developing
                                # doesn't get mistaken for the player's own
                                # motion finishing its crossing

In [ ]:
def _smoothed_angle_change(dir_series, window=5):
    """Same circular-smoothing approach as detect_route_break."""
    dir_rad = np.radians(dir_series.ffill().bfill())
    sin_mean = np.sin(dir_rad).rolling(window, center=True, min_periods=1).mean()
    cos_mean = np.cos(dir_rad).rolling(window, center=True, min_periods=1).mean()

    smoothed_dir = np.degrees(np.arctan2(sin_mean, cos_mean)) % 360
    angle_diff = np.abs(np.diff(smoothed_dir, prepend=smoothed_dir.iloc[0]))
    angle_diff = np.where(angle_diff > 180, 360 - angle_diff, angle_diff)
    
    return angle_diff

def _net_signed_turning(dir_series):
    """Net signed rotation (degrees) over the course of the series -- how
    much the heading has wound around in one consistent rotational
    direction. Each frame-to-frame change is wrapped into (-180, 180] before
    summing, so a genuine loop accumulates toward +/-360 while a simple
    back-and-forth (heading swings one way then reverses) cancels back
    toward 0 instead of adding up."""
    dir_vals = dir_series.ffill().bfill().values
    diffs = np.diff(dir_vals)
    diffs = (diffs + 180) % 360 - 180
    return diffs.sum()

def _find_motion_end(p_df, start_idx):
    """Return the index where the motion segment ends: whichever comes LAST of
    (b) speed drops below threshold and stays there, and
    (c) heading stabilizes (player has settled into a route direction) --
    unless (a) one second (10 frames) after the ball_snap event comes after
    both of those, in which case (a) is used instead. The result can never
    extend past one second after the snap, NOR past the start of a second,
    separate motion by the same player -- (b) and (c) are only ever searched
    for within that window."""
    n = len(p_df)

    # FIX: a player can be tagged 'man_in_motion' more than once on the same
    # play (e.g. a return-style shift followed later by a completely separate
    # jet motion into a handoff). Without this cap, the first motion's window
    # could run straight through the second one, splicing two distinct
    # motions into a single, misleading trajectory.
    later_motion_tags = p_df.index[(p_df['event'] == 'man_in_motion') & (p_df.index > start_idx)]
    next_motion_start_idx = later_motion_tags[0] if len(later_motion_tags) else None

    # (a) one second (10 frames) after the ball_snap event -- also the hard
    # upper bound for the whole search below, so motion tracking can't keep
    # running well past the snap.
    snap_matches = p_df.index[p_df['event'] == 'ball_snap']
    snap_event_idx = snap_matches[0] if len(snap_matches) else None
    one_sec_after_snap = min(snap_event_idx + 10, n - 1) if snap_event_idx is not None else n - 1
    if next_motion_start_idx is not None:
        one_sec_after_snap = min(one_sec_after_snap, next_motion_start_idx - 1)

    # If the play goes live (handoff, forward pass, etc.) before this window
    # would otherwise end, cap it there -- UNLESS the player had already
    # reached genuine motion speed before the snap, in which case a live
    # event firing a beat before the numeric crossing finishes shouldn't
    # retroactively erase a motion that was clearly already well underway.
    pre_snap_window_end = min(one_sec_after_snap, snap_event_idx) if snap_event_idx is not None else one_sec_after_snap
    pre_snap_max_speed = p_df['s'].iloc[start_idx:pre_snap_window_end + 1].max() if pre_snap_window_end >= start_idx else 0

    live_event_matches = p_df.index[
        p_df['event'].isin(PLAY_LIVE_EVENTS) & (p_df.index > start_idx) & (p_df.index <= one_sec_after_snap)
    ]
    if len(live_event_matches) and pre_snap_max_speed < JET_SPEED_THRESH:
        one_sec_after_snap = min(one_sec_after_snap, live_event_matches[0])

    search_end = one_sec_after_snap + 1  # exclusive upper bound for range()

    # (b) speed-stop index: first run of STOP_MIN_FRAMES consecutive frames
    # after start_idx (and no later than one second after the snap) where
    # s < STOP_SPEED_THRESH
    stop_idx = None
    speeds = p_df['s'].values
    running_max = 0.0
    run = 0

    for i in range(start_idx, search_end):
        current_speed = speeds[i]

        # 1. Update running peak speed reached during/after motion start
        if current_speed > running_max:
            running_max = current_speed
            run = 0  # Reset run count whenever a new peak is hit

        # 2. Threshold is 20% off the peak (80% of max speed)
        drop_threshold = running_max * 0.80

        # 3. Check if current speed drops below the threshold
        if running_max > 0 and current_speed < drop_threshold:
            run += 1
            if run >= STOP_MIN_FRAMES:
                stop_idx = i
                break
        else:
            run = 0

    # (c) route-stabilize index: first run of ROUTE_STABLE_FRAMES consecutive
    # frames after start_idx (and no later than one second after the snap)
    # where angle_change_rate stays below threshold
    angle_change = _smoothed_angle_change(p_df['dir'])
    route_idx = None
    run = 0
    for i in range(start_idx, search_end):
        if angle_change[i] < ROUTE_ANGLE_THRESH:
            run += 1
            if run >= ROUTE_STABLE_FRAMES:
                route_idx = i
                break
        else:
            run = 0

    # Motion ends at whichever of (b)/(c) comes LAST -- unless (a) comes after
    # both of those, in which case default to (a). Since (b)/(c) are only ever
    # searched for up through (a), the result can never extend past it.
    bc_candidates = [idx for idx in (stop_idx, route_idx) if idx is not None]

    if bc_candidates:
        last_of_bc = max(bc_candidates)
        if one_sec_after_snap > last_of_bc:
            # (a) comes after both (b) and (c) -- default to (a)
            return one_sec_after_snap
        return last_of_bc

    # Neither (b) nor (c) was detected within the one-second window -- fall back to (a)
    return one_sec_after_snap

def _collapse_continuous_motion_tags(p_df, start_indices):
    """Merge consecutive 'man_in_motion' tags into one if the player never
    actually came to a stop between them -- i.e. it's really one continuous
    motion that got tagged more than once (e.g. a brief re-tag partway
    through a return), not two genuinely distinct motions. A tag only counts
    as starting a new, separate motion if the player had a real stop (a run
    of STOP_MIN_FRAMES consecutive frames below STOP_SPEED_THRESH -- the
    same bar used elsewhere to detect 'stopped') sometime after the previous
    tag and before this one."""
    start_indices = list(start_indices)
    if len(start_indices) <= 1:
        return np.array(start_indices)

    speeds = p_df['s'].values
    collapsed = [start_indices[0]]

    for idx in start_indices[1:]:
        prev_idx = collapsed[-1]

        run = 0
        had_stop = False
        for i in range(prev_idx, idx + 1):
            if speeds[i] < STOP_SPEED_THRESH:
                run += 1
                if run >= STOP_MIN_FRAMES:
                    had_stop = True
                    break
            else:
                run = 0

        if had_stop:
            collapsed.append(idx)  # genuinely a new, separate motion
        # else: just a re-tag of the same continuous motion -- skip it, keep
        # treating this as a continuation of the motion starting at prev_idx

    # FIX: return a numpy array, not a plain list -- callers (classify_motions,
    # debug_classify_motion) do boolean-mask indexing on this result
    # (start_indices > start_idx), which lists don't support.
    return np.array(collapsed)

In [ ]:
def _classify_single_motion(p_df, start_idx, end_idx, ball_x, ball_y,
                            direction_mult, qb_rows):
    """p_df is one player's full-play frames, sorted by frame_id.
    Returns a motion category string."""
    motion_df = p_df.iloc[start_idx:end_idx + 1].reset_index(drop=True)
    if len(motion_df) < MIN_MOTION_FRAMES:
        return None  # too short to classify meaningfully

    start_x, start_y = motion_df['x'].iloc[0], motion_df['y'].iloc[0]

    # --- 1. Return-Style Motion Checks (goes out, comes back near start) ---
    end_x_tight, end_y_tight = motion_df['x'].iloc[-1], motion_df['y'].iloc[-1]
    net_disp = float(np.hypot(end_x_tight - start_x, end_y_tight - start_y))

    dx = motion_df['x'].diff()
    dy = motion_df['y'].diff()
    motion_df['dist_per_frame'] = np.sqrt(dx**2 + dy**2)
    total_distance = motion_df['dist_per_frame'].sum()

    # --- 3. RB Shift Special Case ---
    qb_x, qb_y = None, None
    if not qb_rows.empty and 'ball_snap' in qb_rows['event'].values:
        qb_snap = qb_rows[qb_rows['event'] == 'ball_snap'].iloc[-1]
        qb_x, qb_y = qb_snap['x'], qb_snap['y']

        dist_start_from_qb = np.hypot(start_x - qb_x, start_y - qb_y)
        dist_end_from_qb = np.hypot(end_x_tight - qb_x, end_y_tight - qb_y)

        if dist_start_from_qb <= 3.0 and dist_end_from_qb <= 3.0:
            return 'Hop'

    if net_disp <= RETURN_DIST_THRESH and total_distance >= RETURN_TRAVEL_DIST:
        return 'Return'

    # --- 1b. Return Orbit Check ---
    if not qb_rows.empty and 'ball_snap' in qb_rows['event'].values:
        qb_snap_for_return = qb_rows[qb_rows['event'] == 'ball_snap'].iloc[-1]
        qb_x_for_return, qb_y_for_return = qb_snap_for_return['x'], qb_snap_for_return['y']

        x_rel = motion_df['x'] - qb_x_for_return
        y_rel = motion_df['y'] - qb_y_for_return

        visited_pos_x_pos_y = ((x_rel > 0) & (y_rel > 0)).any()
        visited_pos_x_neg_y = ((x_rel > 0) & (y_rel < 0)).any()
        visited_neg_x_pos_y = ((x_rel < 0) & (y_rel > 0)).any()
        visited_neg_x_neg_y = ((x_rel < 0) & (y_rel < 0)).any()

        looped_around_qb = (
            visited_pos_x_pos_y and visited_pos_x_neg_y and
            visited_neg_x_pos_y and visited_neg_x_neg_y
        )

        if looped_around_qb:
            return 'Orbit Loop'

    # --- 2. Crossing Determination ---
    snap_matches = p_df.index[p_df['event'] == 'ball_snap']
    snap_event_idx = snap_matches[0] if len(snap_matches) else end_idx
    natural_window_end = min(len(p_df) - 1, snap_event_idx + CROSSING_WINDOW_FRAMES)

    later_motion_tags = p_df.index[(p_df['event'] == 'man_in_motion') & (p_df.index > start_idx)]
    next_motion_start_idx = later_motion_tags[0] if len(later_motion_tags) else None
    if next_motion_start_idx is not None:
        natural_window_end = min(natural_window_end, next_motion_start_idx - 1)

    pre_snap_window_end = min(natural_window_end, snap_event_idx) if snap_event_idx is not None else natural_window_end
    pre_snap_max_speed = p_df['s'].iloc[start_idx:pre_snap_window_end + 1].max() if pre_snap_window_end >= start_idx else 0

    live_event_matches = p_df.index[
        p_df['event'].isin(PLAY_LIVE_EVENTS) & (p_df.index > start_idx) & (p_df.index <= natural_window_end)
    ]
    if len(live_event_matches) and pre_snap_max_speed < JET_SPEED_THRESH:
        natural_window_end = min(natural_window_end, live_event_matches[0])

    speeds = p_df['s'].values
    sustained_stop_idx = None
    run = 0
    for i in range(start_idx, natural_window_end + 1):
        if speeds[i] < STOP_SPEED_THRESH:
            run += 1
            if run >= SUSTAINED_STOP_FRAMES:
                sustained_stop_idx = i
                break
        else:
            run = 0

    crossing_window_end = natural_window_end
    if sustained_stop_idx is not None:
        crossing_window_end = min(crossing_window_end, sustained_stop_idx)

    crossing_check_df = p_df.iloc[start_idx:crossing_window_end + 1].reset_index(drop=True)

    end_x, end_y = crossing_check_df['x'].iloc[-1], crossing_check_df['y'].iloc[-1]

    start_y_rel = start_y - ball_y
    end_y_rel = end_y - ball_y
    starting_side_sign = 1 if start_y_rel > 0 else -1

    y_travel = (start_y - end_y) if starting_side_sign > 0 else (end_y - start_y)

    tb_edge_y = ball_y + (starting_side_sign * TACKLE_BOX_OFFSET)
    if starting_side_sign > 0:
        made_it_to_tackle_box = (crossing_check_df['y'] <= tb_edge_y).any()
    else:
        made_it_to_tackle_box = (crossing_check_df['y'] >= tb_edge_y).any()

    if starting_side_sign > 0:
        ever_crosses_ball = (crossing_check_df['y'] < ball_y).any()
    else:
        ever_crosses_ball = (crossing_check_df['y'] > ball_y).any()

    crossed = (
        made_it_to_tackle_box
        and ever_crosses_ball
        and (y_travel >= 3.5)
        and abs(start_y_rel) >= MIN_STARTING_OFFSET_FOR_CROSSING
    )

    # --- 4. Final Classification Flow ---
    if crossed:
        depth_rel = (crossing_check_df['x'] - ball_x) * direction_mult
        max_speed = crossing_check_df['s'].max()

        if starting_side_sign > 0:
            cross_frame_matches = crossing_check_df.index[crossing_check_df['y'] < ball_y]
        else:
            cross_frame_matches = crossing_check_df.index[crossing_check_df['y'] > ball_y]
        first_cross_idx = cross_frame_matches[0] if len(cross_frame_matches) else len(crossing_check_df) - 1

        speeds_in_window = crossing_check_df['s'].values[:first_cross_idx + 1]
        run = 0
        had_brief_stop = False
        for spd in speeds_in_window:
            if spd < STOP_SPEED_THRESH:
                run += 1
                if run >= BRIEF_STOP_FRAMES:
                    had_brief_stop = True
                    break
            else:
                run = 0

        if qb_x is not None and max_speed >= JET_SPEED_THRESH and not had_brief_stop:
            start_depth_behind_los = (start_x - ball_x) * -direction_mult
            started_in_backfield = start_depth_behind_los > BACKFIELD_DEPTH_THRESHOLD
            if dist_start_from_qb <= 2.5 and started_in_backfield:
                return 'Bolt'


            depth_behind_qb = (crossing_check_df['x'] - qb_x) * (-direction_mult)
            y_dist_from_qb = (crossing_check_df['y'] - qb_y).abs()
            is_behind_qb = (
                (depth_behind_qb > IS_BEHIND_QB_THRESHOLD) &
                (y_dist_from_qb <= QB_Y_ALIGNMENT_THRESHOLD)
            ).any()

            qb_alignment = p_df['qbAlignment'].iloc[0] if 'qbAlignment' in p_df.columns else None
            qb_under_center = qb_alignment in ('UNDER CENTER', 'UNDER_CENTER')

            ball_snap_in_window = 'ball_snap' in crossing_check_df['event'].values
            if ball_snap_in_window and p_df[p_df['event'] == 'ball_snap']['s'].iloc[0] > JET_SPEED_THRESH - 1.5:
                return 'Orbit' if is_behind_qb and not qb_under_center else 'Jet'
            else:
                start_depth_behind_los = (start_x - ball_x) * -direction_mult
                started_in_backfield = start_depth_behind_los > BACKFIELD_DEPTH_THRESHOLD
                if dist_start_from_qb <= 2.5 and started_in_backfield:
                    return 'Exit'
                else:
                    return 'Across'

        return 'Across'

    else:
        if abs(end_y_rel) < abs(start_y_rel):
            return 'Short'
        else:
            if not qb_rows.empty and 'ball_snap' in qb_rows['event'].values:
                qb_snap = qb_rows[qb_rows['event'] == 'ball_snap'].iloc[-1]
                qb_x, qb_y = qb_snap['x'], qb_snap['y']

                start_depth_behind_los = (start_x - ball_x) * -direction_mult
                started_in_backfield = start_depth_behind_los > BACKFIELD_DEPTH_THRESHOLD

                if dist_start_from_qb <= 4.0 and started_in_backfield:
                    return 'Exit'
                else:
                    return 'Out'
            else:
                return 'Out'

In [ ]:
def classify_motions(df, game_id, gsis_play_id):
    """Returns {(nflId, motion_index): {'category': str, 'start_frame': int,
    'end_frame': int, ...}} -- one entry per distinct motion. A player tagged
    'man_in_motion' more than once on the same play (e.g. a return-style
    shift followed later by a completely separate jet motion into a handoff)
    gets one entry per occurrence, each classified independently on its own
    start/end -- motion_index counts up from 0 in chronological order for
    that player."""
    play_df = df[(df['game_id'] == game_id) & (df['gsis_play_id'] == gsis_play_id)].copy()
    if play_df.empty:
        raise ValueError(f"No rows found for game {game_id}, play {gsis_play_id}")

    ball_x = play_df['x_ball_at_snap'].iloc[0]
    ball_y = play_df['y_ball_at_snap'].iloc[0]
    direction_mult = -1 if play_df['play_direction'].iloc[0] == 'left' else 1

    # QB depth reference: average QB x position relative to LOS, in downfield-positive convention
    qb_rows = play_df[play_df['position_group'] == 'QB']

    motion_player_ids = play_df.loc[play_df['event'] == 'man_in_motion', 'nflId'].unique()

    results = {}
    for nflId in motion_player_ids:
        p_df = play_df[play_df['nflId'] == nflId].sort_values('frame_id').reset_index(drop=True)

        if p_df['position_group'].iloc[0] not in ['WR', 'RB', 'TE']:
            continue

        raw_start_indices = p_df.index[p_df['event'] == 'man_in_motion']
        if len(raw_start_indices) == 0:
            continue

        # Multiple 'man_in_motion' tags only count as separate motions if the
        # player genuinely stopped in between -- otherwise it's the same
        # continuous motion re-tagged, not two distinct ones.
        start_indices = _collapse_continuous_motion_tags(p_df, raw_start_indices)

        for motion_index, start_idx in enumerate(start_indices):
            # FILTER: 'man_in_motion' is a frame-level event tag -- every player's
            # row at that frame_id gets it, not just the player who actually went
            # in motion. Require this specific player to have actually hit a real
            # speed sometime between this tag and the next boundary (the snap, or
            # the start of their next motion, whichever comes first) to count as
            # genuinely being in motion for this occurrence.
            later_start_indices = start_indices[start_indices > start_idx]
            next_motion_start_idx = later_start_indices[0] if len(later_start_indices) else None

            snap_matches = p_df.index[p_df['event'] == 'ball_snap']
            snap_event_idx = snap_matches[0] if len(snap_matches) else None
            pre_snap_end = snap_event_idx if snap_event_idx is not None else start_idx
            if next_motion_start_idx is not None:
                pre_snap_end = min(pre_snap_end, next_motion_start_idx - 1)
            pre_snap_segment = p_df.iloc[start_idx:pre_snap_end + 1]
            if pre_snap_segment.empty or pre_snap_segment['s'].max() < MOTION_MIN_SPEED:
                continue

            end_idx = _find_motion_end(p_df, start_idx)
            motion_avg_speed = round(p_df.iloc[start_idx:end_idx + 1]['s'].mean())
            if motion_avg_speed <= 0:
                continue

            category = _classify_single_motion(
                p_df, start_idx, end_idx, ball_x, ball_y, direction_mult, qb_rows
            )
            if category is None:
                continue

            results[(int(nflId), motion_index)] = {
                'category': category,
                'game_id': p_df['game_id'].iloc[start_idx],
                'gsis_play_id': p_df['gsis_play_id'].iloc[start_idx],
                'player_name': p_df['displayName'].iloc[start_idx],
                'nflId': int(nflId),
                'motion_index': motion_index,
                'start_frame': p_df['frame_id'].iloc[start_idx],
                'end_frame': p_df['frame_id'].iloc[end_idx],
            }

    return results

In [ ]:
def debug_classify_motion(df, game_id, gsis_play_id, nflId, motion_index=0):
    """Walks through the exact same logic as _classify_single_motion for one
    player on one play, printing every intermediate value along the way --
    use this on a play that's still misclassified to see exactly which
    condition is responsible, instead of guessing.

    game_id, gsis_play_id: strings, matching motion_tracking_data's dtype.
    nflId: int.
    motion_index: which 'man_in_motion' occurrence to debug if this player has
        more than one on this play (0 = first, 1 = second, etc.) -- matches
        the motion_index half of classify_motions' (nflId, motion_index) keys.
    """
    play_df = df[(df['game_id'] == game_id) & (df['gsis_play_id'] == gsis_play_id)].copy()
    if play_df.empty:
        print(f"No rows found for game {game_id}, play {gsis_play_id}")
        return

    ball_x = play_df['x_ball_at_snap'].iloc[0]
    ball_y = play_df['y_ball_at_snap'].iloc[0]
    direction_mult = -1 if play_df['play_direction'].iloc[0] == 'left' else 1
    qb_rows = play_df[play_df['position_group'] == 'QB']

    p_df = play_df[play_df['nflId'] == nflId].sort_values('frame_id').reset_index(drop=True)
    if p_df.empty:
        print(f"No rows found for nflId {nflId} on this play")
        return

    raw_start_indices = p_df.index[p_df['event'] == 'man_in_motion']
    if len(raw_start_indices) == 0:
        print("No man_in_motion event tagged for this player")
        return

    start_indices = _collapse_continuous_motion_tags(p_df, raw_start_indices)
    if len(start_indices) < len(raw_start_indices):
        print(f"NOTE: {len(raw_start_indices)} man_in_motion tags at {list(raw_start_indices)} collapsed down "
              f"to {len(start_indices)} genuine motion(s) at {start_indices} (no stop found between some of them)")

    if motion_index >= len(start_indices):
        print(f"This player only has {len(start_indices)} genuine motion(s) after collapsing -- motion_index={motion_index} is out of range")
        return
    start_idx = start_indices[motion_index]
    if len(start_indices) > 1:
        print(f"NOTE: this player has {len(start_indices)} genuine motions at indices {start_indices} -- debugging motion_index={motion_index} (start_idx={start_idx})")

    end_idx = _find_motion_end(p_df, start_idx)

    snap_matches = p_df.index[p_df['event'] == 'ball_snap']
    snap_event_idx = snap_matches[0] if len(snap_matches) else end_idx
    natural_window_end = min(len(p_df) - 1, snap_event_idx + CROSSING_WINDOW_FRAMES)

    later_motion_tags = p_df.index[(p_df['event'] == 'man_in_motion') & (p_df.index > start_idx)]
    next_motion_start_idx = later_motion_tags[0] if len(later_motion_tags) else None
    if next_motion_start_idx is not None:
        natural_window_end = min(natural_window_end, next_motion_start_idx - 1)
        print(f"NOTE: a second 'man_in_motion' tag exists at index {next_motion_start_idx} -- window capped there")

    pre_snap_window_end = min(natural_window_end, snap_event_idx) if snap_event_idx is not None else natural_window_end
    pre_snap_max_speed = p_df['s'].iloc[start_idx:pre_snap_window_end + 1].max() if pre_snap_window_end >= start_idx else 0

    live_event_matches = p_df.index[
        p_df['event'].isin(PLAY_LIVE_EVENTS) & (p_df.index > start_idx) & (p_df.index <= natural_window_end)
    ]
    if len(live_event_matches):
        if pre_snap_max_speed < JET_SPEED_THRESH:
            natural_window_end = min(natural_window_end, live_event_matches[0])
            print(f"NOTE: a play-live event ({p_df['event'].iloc[live_event_matches[0]]}) exists at index {live_event_matches[0]} -- window capped there (pre_snap_max_speed={pre_snap_max_speed:.2f} < JET_SPEED_THRESH)")
        else:
            print(f"NOTE: a play-live event ({p_df['event'].iloc[live_event_matches[0]]}) exists at index {live_event_matches[0]} but pre_snap_max_speed={pre_snap_max_speed:.2f} >= JET_SPEED_THRESH -- window NOT capped there")

    speeds = p_df['s'].values
    sustained_stop_idx = None
    run = 0
    for i in range(start_idx, natural_window_end + 1):
        if speeds[i] < STOP_SPEED_THRESH:
            run += 1
            if run >= SUSTAINED_STOP_FRAMES:
                sustained_stop_idx = i
                break
        else:
            run = 0

    crossing_window_end = natural_window_end
    if sustained_stop_idx is not None:
        crossing_window_end = min(crossing_window_end, sustained_stop_idx)

    crossing_check_df = p_df.iloc[start_idx:crossing_window_end + 1].reset_index(drop=True)
    motion_df = p_df.iloc[start_idx:end_idx + 1].reset_index(drop=True)

    print(f"ball_x={ball_x:.2f}, ball_y={ball_y:.2f}, direction_mult={direction_mult}")
    print(f"len(p_df)={len(p_df)}, start_idx={start_idx}, end_idx={end_idx}, "
          f"snap_event_idx={snap_event_idx}, natural_window_end={natural_window_end}, "
          f"sustained_stop_idx={sustained_stop_idx}, crossing_window_end={crossing_window_end}, "
          f"len(crossing_check_df)={len(crossing_check_df)}")

    start_x, start_y = motion_df['x'].iloc[0], motion_df['y'].iloc[0]
    end_x, end_y = crossing_check_df['x'].iloc[-1], crossing_check_df['y'].iloc[-1]
    print(f"start=({start_x:.2f}, {start_y:.2f}), end (at crossing window)=({end_x:.2f}, {end_y:.2f})")

    start_y_rel = start_y - ball_y
    end_y_rel = end_y - ball_y
    starting_side_sign = 1 if start_y_rel > 0 else -1
    y_travel = (start_y - end_y) if starting_side_sign > 0 else (end_y - start_y)
    print(f"start_y_rel={start_y_rel:.2f}, end_y_rel={end_y_rel:.2f}, "
          f"starting_side_sign={starting_side_sign}, y_travel={y_travel:.2f} "
          f"(needs >= 3.5)")

    tb_edge_y = ball_y + (starting_side_sign * TACKLE_BOX_OFFSET)
    if starting_side_sign > 0:
        made_it_to_tackle_box = (crossing_check_df['y'] <= tb_edge_y).any()
        ever_crosses_ball = (crossing_check_df['y'] < ball_y).any()
    else:
        made_it_to_tackle_box = (crossing_check_df['y'] >= tb_edge_y).any()
        ever_crosses_ball = (crossing_check_df['y'] > ball_y).any()
    print(f"tb_edge_y={tb_edge_y:.2f}, made_it_to_tackle_box={made_it_to_tackle_box}, "
          f"ever_crosses_ball={ever_crosses_ball}")

    crossed = (
        made_it_to_tackle_box
        and ever_crosses_ball
        and (y_travel >= 3.5)
        and abs(start_y_rel) >= MIN_STARTING_OFFSET_FOR_CROSSING
    )
    print(f"crossed={crossed} (abs(start_y_rel)={abs(start_y_rel):.2f}, needs >= {MIN_STARTING_OFFSET_FOR_CROSSING})")

    print()
    print("y trajectory within crossing_check_df (frame_id, x, y, s, event):")
    print(crossing_check_df[['frame_id', 'x', 'y', 's', 'event']].to_string())

    qb_x, qb_y = None, None
    if not qb_rows.empty and 'ball_snap' in qb_rows['event'].values:
        qb_snap = qb_rows[qb_rows['event'] == 'ball_snap'].iloc[-1]
        qb_x, qb_y = qb_snap['x'], qb_snap['y']

    if crossed:
        max_speed = crossing_check_df['s'].max()

        if starting_side_sign > 0:
            cross_frame_matches = crossing_check_df.index[crossing_check_df['y'] < ball_y]
        else:
            cross_frame_matches = crossing_check_df.index[crossing_check_df['y'] > ball_y]
        first_cross_idx = cross_frame_matches[0] if len(cross_frame_matches) else len(crossing_check_df) - 1

        speeds_in_window = crossing_check_df['s'].values[:first_cross_idx + 1]
        run = 0
        had_brief_stop = False
        for spd in speeds_in_window:
            if spd < STOP_SPEED_THRESH:
                run += 1
                if run >= BRIEF_STOP_FRAMES:
                    had_brief_stop = True
                    break
            else:
                run = 0

        print(f"first_cross_idx={first_cross_idx} (within crossing_check_df), "
              f"max_speed={max_speed:.2f} (needs >= JET_SPEED_THRESH={JET_SPEED_THRESH} for Jet/Orbit), "
              f"had_brief_stop={had_brief_stop} (checked only up through first_cross_idx; forces Across if True)")

        if qb_x is not None and max_speed >= JET_SPEED_THRESH and not had_brief_stop:
            depth_behind_qb = (crossing_check_df['x'] - qb_x) * (-direction_mult)
            y_dist_from_qb = (crossing_check_df['y'] - qb_y).abs()
            behind_and_aligned = (
                (depth_behind_qb > IS_BEHIND_QB_THRESHOLD) &
                (y_dist_from_qb <= QB_Y_ALIGNMENT_THRESHOLD)
            )
            is_behind_qb = behind_and_aligned.any()
            max_depth_behind_qb = depth_behind_qb.max()
            depth_behind_qb_frame = crossing_check_df.iloc[depth_behind_qb.values.argmax()]['frame_id'] if len(depth_behind_qb) else None
            min_y_dist_when_deep = y_dist_from_qb[depth_behind_qb > IS_BEHIND_QB_THRESHOLD].min() if (depth_behind_qb > IS_BEHIND_QB_THRESHOLD).any() else None

            qb_alignment = p_df['qbAlignment'].iloc[0] if 'qbAlignment' in p_df.columns else None
            qb_under_center = qb_alignment in ('UNDER CENTER', 'UNDER_CENTER')

            snap_speed = p_df[p_df['event'] == 'ball_snap']['s'].iloc[0]

            print(f"qb_x={qb_x:.2f}, qb_y={qb_y:.2f}, qb_alignment={qb_alignment!r}, qb_under_center={qb_under_center}")
            min_y_dist_str = f"{min_y_dist_when_deep:.2f}" if min_y_dist_when_deep is not None else "N/A"
            print(f"max_depth_behind_qb={max_depth_behind_qb:.2f} (needs > {IS_BEHIND_QB_THRESHOLD}), "
                  f"closest y-alignment while deep enough={min_y_dist_str} (needs <= {QB_Y_ALIGNMENT_THRESHOLD}), "
                  f"is_behind_qb={is_behind_qb}")
            print(f"snap_speed={snap_speed:.2f} (needs > {JET_SPEED_THRESH - 1.5} to even consider Orbit/Jet split)")

            if snap_speed > JET_SPEED_THRESH - 1.5:
                final_category = 'Orbit' if is_behind_qb and not qb_under_center else 'Jet'
            else:
                final_category = 'Across'
            print(f"FINAL: {final_category}")
        else:
            print("FINAL: Across (qb_x missing, max_speed < JET_SPEED_THRESH, or had_brief_stop)")

    return {
        'start_idx': start_idx, 'end_idx': end_idx, 'snap_event_idx': snap_event_idx,
        'crossing_window_end': crossing_window_end, 'y_travel': y_travel,
        'made_it_to_tackle_box': made_it_to_tackle_box, 'ever_crosses_ball': ever_crosses_ball,
        'crossed': crossed,
    }

In [ ]:
for keys, group in tracking_data.groupby(play_keys):
    gameId, gsisPlayId = keys

    # FIX: `group` is every frame this play has for a given player unless it's
    # explicitly cut down here -- without this, qb_df/ol_df/backfield_df below
    # count the same player multiple times (once per frame) instead of once
    # each at the snap, which is what was throwing off backfield_count.
    group = group[group['event'] == 'line_set'].copy()
    if group.empty:
        print(f"WARNING: no line_set frame for game {gameId}, play {gsisPlayId} -- skipping")
        continue

    play_row = group.iloc[0]

    qb_df = group[group['position_group'] == 'QB']
    if len(qb_df) > 0:
        qb_x = round(qb_df['x_player_ball_norm_at_line_set'].iloc[0], 2)
        qb_y = round(qb_df['y_player_ball_norm_at_line_set'].iloc[0], 2)  # BUG FIX: was reading x-column for qb_y
    else:
        qb_x, qb_y = None, None

    ol_df = group[group['position_group'] == 'OL']
    backfield_depth_threshold = ol_df['x_player_ball_norm_at_line_set'].mean()
    qb_dir = 'positive' if qb_x is not None and qb_x < backfield_depth_threshold else 'negative'

    # FIX: this used to re-derive the play direction with a *different*
    # comparison (qb_x < 0) than the one that set qb_dir (qb_x <
    # backfield_depth_threshold). backfield_depth_threshold is almost never
    # exactly 0 -- the OL doesn't sit precisely on top of the ball -- so for
    # QBs close to the line (e.g. under center), the two comparisons could
    # disagree and pick the wrong mask direction, which silently dropped real
    # backfield players (RB/FB/etc.) from backfield_df_raw. Reuse qb_dir
    # itself everywhere instead of re-deriving it.
    qb_dir_positive = qb_dir == 'positive'

    if qb_dir_positive:
        backfield_mask = group['x_player_ball_norm_at_line_set'] < -backfield_depth_threshold
        ol_left_edge = ol_df['y_player_ball_norm_at_line_set'].max()
        ol_right_edge = ol_df['y_player_ball_norm_at_line_set'].min()
    else:
        backfield_mask = group['x_player_ball_norm_at_line_set'] > backfield_depth_threshold
        ol_left_edge = ol_df['y_player_ball_norm_at_line_set'].min()
        ol_right_edge = ol_df['y_player_ball_norm_at_line_set'].max()

    backfield_df_raw = group[
        (group['position_group'].isin(skill_positions)) &
        backfield_mask
    ]

    if qb_dir_positive:
        outside_tackles_mask = (
            (backfield_df_raw['y_player_ball_norm_at_line_set'] > ol_left_edge) |
            (backfield_df_raw['y_player_ball_norm_at_line_set'] < ol_right_edge)
        )
    else:
        outside_tackles_mask = (
            (backfield_df_raw['y_player_ball_norm_at_line_set'] < ol_left_edge) |
            (backfield_df_raw['y_player_ball_norm_at_line_set'] > ol_right_edge)
        )

    backfield_outside_tackles = bool(outside_tackles_mask.any())
    backfield_df = backfield_df_raw[~outside_tackles_mask]
    backfield_count = len(backfield_df)

    if qb_dir_positive:
        dist_from_line = np.where(
            group['y_player_ball_norm_at_line_set'] > 0,
            group['y_player_ball_norm_at_line_set'] - ol_left_edge,
            abs(group['y_player_ball_norm_at_line_set'] - ol_right_edge)
        )
    else:
        dist_from_line = np.where(
            group['y_player_ball_norm_at_line_set'] > 0,
            group['y_player_ball_norm_at_line_set'] - ol_right_edge,
            abs(group['y_player_ball_norm_at_line_set'] - ol_left_edge)
        )

    group = group.copy()
    group['dist_from_line'] = np.round(dist_from_line)

    skill_position_df = group[group['position_group'].isin(skill_positions)]

    te_attached_df = skill_position_df[
        (skill_position_df['position_group'] == 'TE') &
        (np.trunc(skill_position_df['x_player_ball_norm_at_line_set']) <= backfield_depth_threshold) &
        (skill_position_df['dist_from_line'].abs() <= 2)
    ]
    te_attached_pos = (te_attached_df['y_player_ball_norm_at_line_set'] > 0).sum()
    te_attached_neg = (te_attached_df['y_player_ball_norm_at_line_set'] < 0).sum()

    te_on_los_df = skill_position_df[
        (skill_position_df['position_group'] == 'TE') &
        (skill_position_df['x_player_ball_norm_at_line_set'].abs() <= 0.25)
    ]
    te_on_los_pos = (te_on_los_df['y_player_ball_norm_at_line_set'] > 0).sum()
    te_on_los_neg = (te_on_los_df['y_player_ball_norm_at_line_set'] < 0).sum()

    te_all_df = skill_position_df[skill_position_df['position_group'] == 'TE']
    te_all_pos = (te_all_df['y_player_ball_norm_at_line_set'] > 0).sum()
    te_all_neg = (te_all_df['y_player_ball_norm_at_line_set'] < 0).sum()

    receiver_df = skill_position_df[skill_position_df['position_group'].isin(['WR', 'TE'])]
    receiver_pos = (receiver_df['y_player_ball_norm_at_line_set'] > 0).sum()
    receiver_neg = (receiver_df['y_player_ball_norm_at_line_set'] < 0).sum()

    field_width = 53.33
    y_ball = group['y_ball_at_snap'].iloc[0]
    field_pos_side = 'positive' if y_ball < field_width / 2 else 'negative'

    formation_strength = (
        resolve_side(te_attached_pos, te_attached_neg)
        or resolve_tier2(te_on_los_pos, te_on_los_neg)
        or resolve_side(te_all_pos, te_all_neg)
        or resolve_side(receiver_pos, receiver_neg)
        or field_pos_side
    )

    players = []
    for _, row in group.iterrows():
        players.append({
            'nflId': row['nflId'],
            'displayName': row['displayName'],
            'position_group': row['position_group'],
            'x_at_line_set': row['x'],
            'y_at_line_set': row['y'],
            'x_at_snap': row['xAtSnap'],
            'y_at_snap': row['yAtSnap'],
            'x_diff_from_snap': safe_round(row['x_diff_from_snap']),
            'y_diff_from_snap': safe_round(row['y_diff_from_snap']),
            'x_player_ball_norm_at_snap': safe_round(row['x_player_ball_norm_at_line_set'], 2),
            'y_player_ball_norm_at_snap': safe_round(row['y_player_ball_norm_at_line_set'], 2),
            'x_player_ball_norm_at_line_set': safe_round(row['x_ball_at_snap'] - row['x'], 2),
            'y_player_ball_norm_at_line_set': safe_round(row['y_ball_at_snap'] - row['y'], 2),
        })

    entry = {
        'gameId': gameId,
        'gsisPlayId': gsisPlayId,
        'teamId': play_row['teamId'],
        'qb_alignment': play_row['qbAlignment'],
        'x_ball_at_snap': play_row['x_ball_at_snap'],
        'y_ball_at_snap': play_row['y_ball_at_snap'],
        'players': players,
        'qb_locations': qb_df,
        'backfield_locations': backfield_df,
        'qb_dir': qb_dir,
        'backfield_depth': backfield_depth_threshold,
        'backfield_outside_tackles': backfield_outside_tackles,
        'backfield_count': backfield_count,
        'ol_left_edge': safe_round(ol_left_edge, 2),
        'ol_right_edge': safe_round(ol_right_edge, 2),
        'formation_strength': formation_strength,
        'team_in_trips': play_row['teamReceiverFormation.trips'],
        'sniffer': play_row['teamOffenseFormationBase.hasSniffer'],  # renamed source column
        # 'on_los': NOT AVAILABLE in this dataset — no equivalent column
        'left_count': play_row['teamReceiverFormation.leftCount'],
        'right_count': play_row['teamReceiverFormation.rightCount'],
        'tight_left_count': play_row['teamReceiverFormation.tightLeftCount'],
        'tight_right_count': play_row['teamReceiverFormation.tightRightCount'],
        'detached_left_count': play_row['teamReceiverFormation.detachedLeftCount'],
        'detached_right_count': play_row['teamReceiverFormation.detachedRightCount'],
        'bunched_left': play_row['teamReceiverFormation.bunchedLeft'],
        'bunched_right': play_row['teamReceiverFormation.bunchedRight'],
        'bunched': bool(play_row['teamReceiverFormation.bunchedLeft']) or bool(play_row['teamReceiverFormation.bunchedRight']),
        'bundled_left': play_row['teamReceiverFormation.bundledLeft'],
        'bundled_right': play_row['teamReceiverFormation.bundledRight'],
        'bundled': bool(play_row['teamReceiverFormation.bundledLeft']) or bool(play_row['teamReceiverFormation.bundledRight']),
        'formation_raw': play_row['offenseFormation_x'],  # replaces old 'formation_raw' column
    }

    formation_results[keys] = entry

In [ ]:
# Kneel plays never get real offensive-formation data (no offenseFormation,
# no qbAlignment) -- there's no formation being schemed on a QB kneel, so
# empty_backfield/single_backfield/double_backfield have nothing meaningful
# to work with for them. Detect these directly via the 'qb_kneel' event
# (rather than inferring from the NaN fields, which could theoretically
# happen for other reasons) and give them their own explicit label instead
# of running them through the normal backfield-description logic.
kneel_plays = set(
    raw_tracking_data.loc[raw_tracking_data['event'] == 'qb_kneel', ['game_id', 'gsis_play_id']]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

for key, item in formation_results.items():
    current_qb = item['qb_locations']
    current_qb_x = current_qb['x'].iloc[0]
    current_qb_y = current_qb['y'].iloc[0]

    if key in kneel_plays:
        backfield_desc = 'Kneel'
    elif item['backfield_count'] == 0:
        backfield_desc = empty_backfield(item['qb_alignment'])
    elif item['backfield_count'] == 1:
        backfield_desc = single_backfield(item)
    elif item['backfield_count'] == 2:
        backfield_desc = double_backfield(item, current_qb_x, current_qb_y)
    else:
        backfield_desc = '3+ Backfield Set'

    motion_desc = ''  # BUG FIX: was uninitialized when no player moved — would raise NameError or leak the previous play's value
    if backfield_desc and (
        any(p['x_diff_from_snap'] != 0 for p in item['players']) or
        any(p['y_diff_from_snap'] != 0 for p in item['players'])
    ):
        motion_desc = motion_suffix(item['players'])

    avg_locs = average_locations(item['players'], item['backfield_depth'], item['ol_left_edge'], item['ol_right_edge'])
    keys_out = ['average_offense_location', 'average_skill_position_location', 'average_receiver_location',
                'average_backfield_location', 'average_defense_location', 'average_secondary_location',
                'average_front_location']

    formation_results[key]['backfield_desc'] = backfield_desc
    formation_results[key]['backfield_desc_with_motion'] = backfield_desc + motion_desc
    for k, v in zip(keys_out, avg_locs):
        formation_results[key][k] = v

In [ ]:
CATEGORY_COLORS = {
    'Short': 'mediumblue',
    'Out': 'darkorange',
    'Across': 'seagreen',
    'Jet': 'crimson',
    'Orbit': 'purple',
    'Return': 'goldenrod',
    'Orbit Loop': 'teal',
    'Exit': 'gray',
    'Hop': 'darkviolet',
}
 
def plot_motion(df_player, motion_info, play_df=None):
    """df_player: one player's full-play frames, sorted by frame_id.
    motion_info: {'category': str, 'start_frame': int, 'end_frame': int}
    play_df: full play frame data (all players) -- optional, used to plot
    the QB reference position/depth line if position_group is available."""
 
    df_player = df_player.sort_values('frame_id').reset_index(drop=True)
    snap_frame = df_player[df_player['event'] == 'ball_snap']['frame_id'].iloc[0]
    category = motion_info.get('category', 'unknown')
    start_frame = motion_info['start_frame']
    end_frame = motion_info['end_frame']
 
    # Match frame_id to the exact positional index in df_player
    start_matches = df_player.index[df_player['frame_id'] == start_frame]
    end_matches = df_player.index[df_player['frame_id'] == end_frame]
    start_idx = start_matches[0] if len(start_matches) > 0 else 0
    end_idx = end_matches[0] if len(end_matches) > 0 else len(df_player) - 1
 
    los_x = df_player['x_ball_at_snap'].iloc[0]
    los_y = df_player['y_ball_at_snap'].iloc[0]
    play_direction = df_player['play_direction'].iloc[0]
    direction_mult = -1 if str(play_direction).strip().lower() == 'left' else 1
 
    color = CATEGORY_COLORS.get(category, 'steelblue')
 
    fig, (ax_map, ax_speed, ax_rate) = plt.subplots(1, 3, figsize=(19, 6))
 
    # --- Field map ---
    pre = df_player.iloc[:start_idx + 1]
    motion = df_player.iloc[start_idx:end_idx + 1]
    post = df_player.iloc[end_idx:]
    snap_x = df_player[df_player['event'] == 'ball_snap']['x'].iloc[-1]
    snap_y = df_player[df_player['event'] == 'ball_snap']['y'].iloc[-1]
    # print(snap_x)
 
    if len(pre) > 1:
        ax_map.plot(pre['x'], pre['y'], color='lightgray', linewidth=1.5, zorder=1, label='Pre-motion')
    ax_map.plot(motion['x'], motion['y'], color=color, linewidth=2.5, zorder=4, label=f'Motion ({category})')
    if len(post) > 1:
        ax_map.plot(post['x'], post['y'], color='lightgray', linewidth=1.5, linestyle='--', zorder=1, label='Post-motion')

    ax_map.scatter(snap_x, snap_y, color='black', linewidth=1.5, linestyle='--', zorder=1, label='Position at Snap')
    ax_map.scatter(df_player['x'].iloc[start_idx], df_player['y'].iloc[start_idx],
                   color='green', marker='o', s=90, zorder=4, label='Motion start')
    ax_map.scatter(df_player['x'].iloc[end_idx], df_player['y'].iloc[end_idx],
                   color='black', marker='s', s=90, zorder=4, label='Motion end')
    # ax_map.scatter(df_player['x_ball_at_snap'].iloc[end_idx], df_player['x_ball_at_snap'].iloc[end_idx],
    #                color='black', marker='o', s=90, zorder=4, label='Ball')
 
    # LOS + direction arrow (same convention as plot_route_with_breaks)
    if pd.notna(los_x) and pd.notna(los_y):
        arrow_dx = 5 * direction_mult
        ax_map.axvline(los_x, color='gray', linewidth=1, linestyle='--', alpha=0.6, zorder=0, label='Line of scrimmage')
        ax_map.annotate(
            '', xy=(los_x + arrow_dx, los_y), xytext=(los_x, los_y),
            arrowprops=dict(arrowstyle='-|>', color='darkorange', linewidth=2, mutation_scale=20),
            zorder=5
        )
    else:
        arrow_dx = 0
 
    # QB depth reference line -- this is what jet/orbit classification is measured against
    qb_x_plot = None
    qb_y_plot = None
    if play_df is not None and 'position_group' in play_df.columns:
        qb_rows = play_df[play_df['position_group'] == 'QB']
        if not qb_rows.empty:
            qb_x_plot = qb_rows[qb_rows['event'] == 'ball_snap']['x'].iloc[-1]
            qb_y_plot = qb_rows[qb_rows['event'] == 'ball_snap']['y'].iloc[-1]
            ax_map.scatter(qb_x_plot, qb_y_plot, color='purple', marker='x', s=90, zorder=0, label='QB at Snap')
 
    all_x = list(df_player['x']) + [los_x, los_x + arrow_dx]
    all_y = list(df_player['y']) + [los_y]
    if qb_x_plot is not None:
        all_x.append(qb_x_plot)
    pad = 3
    ax_map.set_xlim(min(all_x) - pad, max(all_x) + pad)
    ax_map.set_ylim(min(all_y) - pad, max(all_y) + pad)
    ax_map.set_aspect('equal')
    ax_map.set_title(f"Motion Path — {df_player.get('displayName', pd.Series(['Unknown'])).iloc[0]}\nClassified: {category.upper()}")
    ax_map.legend(loc='best', fontsize=7)
 
    # --- Speed panel (jet threshold + stop threshold) ---
    ax_speed.plot(df_player['frame_id'], df_player['s'], color='steelblue', linewidth=1.2, zorder=1)
    ax_speed.axvspan(start_frame, end_frame, color=color, alpha=0.15, zorder=0, label='Motion segment')
    ax_speed.axvline(snap_frame, color='black', linestyle='--', linewidth=1, alpha=0.7, label='Snap Frame')
    ax_speed.axhline(JET_SPEED_THRESH-1.5, color='black', linestyle='--', linewidth=1, alpha=0.7, label='End jet speed threshold')
    ax_speed.axhline(JET_SPEED_THRESH, color='crimson', linestyle='--', linewidth=1, alpha=0.7, label='Jet speed threshold')
    ax_speed.axhline(STOP_SPEED_THRESH, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Stop threshold')
    ax_speed.set_xlim(df_player['frame_id'].min(), df_player['frame_id'].max())
    ax_speed.xaxis.set_major_locator(MultipleLocator(10))
    ax_speed.set_title("Speed")
    ax_speed.set_xlabel("Frame ID")
    ax_speed.set_ylabel("Yards / sec")
    ax_speed.legend(loc='best', fontsize=7)
 
    # --- Angle-change-rate panel (route-stabilize / return threshold) ---
    angle_change = _smoothed_angle_change(df_player['dir'])
    ax_rate.plot(df_player['frame_id'], angle_change, color='steelblue', linewidth=1.2, zorder=1)
    ax_rate.axvspan(start_frame, end_frame, color=color, alpha=0.15, zorder=0)
    ax_rate.axhline(ROUTE_ANGLE_THRESH, color='seagreen', linestyle='--', linewidth=1, alpha=0.7, label='Route-stabilize threshold')
    ax_rate.set_xlim(df_player['frame_id'].min(), df_player['frame_id'].max())
    ax_rate.xaxis.set_major_locator(MultipleLocator(10))
    ax_rate.set_title("Angle Change Rate")
    ax_rate.set_xlabel("Frame ID")
    ax_rate.set_ylabel("Degrees / frame")
    ax_rate.legend(loc='best', fontsize=7)
 
    plt.tight_layout()
    plt.show()

In [ ]:
STYLE_MAP = {
    'QB': dict(marker='o', facecolor='yellow', edgecolor='black', size=180),
    'RB': dict(marker='o', facecolor='red',    edgecolor='black', size=220),
    'TE': dict(marker='o', facecolor='green',  edgecolor='black', size=200),
    'WR': dict(marker='o', facecolor='blue',   edgecolor='black', size=180),
    'OL': dict(marker='o', facecolor='black',  edgecolor='black', size=250),
    'LB': dict(marker='o', facecolor='orange', edgecolor='black', size=220),
    'DB': dict(marker='o', facecolor='magenta',edgecolor='black', size=180),
    'DL': dict(marker='o', facecolor='black',  edgecolor='black', size=250),
}

DEFAULT_STYLE = dict(marker='o', facecolor='lightgray', edgecolor='gray', size=350)

AVG_STYLE_MAP = {
    'average_offense_location':          dict(marker='x', color='purple',   label='Avg Offense', size=100),
    'average_skill_position_location':   dict(marker='x', color='green',    label='Avg Skill Position', size=100),
    'average_receiver_location':         dict(marker='x', color='dodgerblue', label='Avg Receiver', size=100),
    'average_backfield_location':        dict(marker='x', color='red',     label='Avg Backfield', size=100),
    'average_defense_location':          dict(marker='x', color='purple',  label='Avg Defense', size=100),
    'average_secondary_location':        dict(marker='x', color='dodgerblue', label='Avg Secondary', size=100),
    'average_front_location':            dict(marker='x', color='red',  label='Avg Front', size=100),
}

CONTROL_AREA_GROUPS = {
    'defense_overall': dict(pos_groups=('DL', 'LB', 'DB'), color='purple',  label='Defense Control Area'),
    'front':           dict(pos_groups=('DL', 'LB'),       color='orange', label='Front Control Area'),
    'secondary':       dict(pos_groups=('DB',),            color='magenta', label='Secondary Control Area'),
    # 'offense_overall': dict(pos_groups=('QB', 'OL', 'WR', 'RB', 'TE'), color='navy',   label='Offense Control Area'),
    # 'skill_position':  dict(pos_groups=('WR', 'RB', 'TE'),             color='teal',   label='Skill Position Control Area'),
    # 'receivers':       dict(pos_groups=('WR',),                         color='dodgerblue', label='Receiver Control Area'),
    # 'backfield':       dict(pos_groups=('RB', 'TE'),                    color='crimson', label='Backfield Control Area'),
}

def draw_players(ax, entry, direction_mult):
    for player in entry['players']:
        style = STYLE_MAP.get(player['position_group'], DEFAULT_STYLE)

        px = -direction_mult * player['y_player_ball_norm_at_line_set']
        py = direction_mult * player['x_player_ball_norm_at_line_set']

        moved = player['x_diff_from_snap'] != 0 or player['y_diff_from_snap'] != 0

        ax.scatter(px, py, s=style['size'], marker=style['marker'],
                facecolors=style['facecolor'], edgecolors=style['edgecolor'],
                linewidths=1.5, alpha=0.1 if moved else 1.0, zorder=2 if moved else 3)

        if moved:
            snap_px = -direction_mult * player['y_player_ball_norm_at_snap']
            snap_py = direction_mult * player['x_player_ball_norm_at_snap']

            ax.scatter(snap_px, snap_py, s=style['size'], marker=style['marker'],
                    facecolors=style['facecolor'], edgecolors=style['edgecolor'],
                    linewidths=1.5, alpha=1.0, zorder=3)

            ax.plot([px, snap_px], [py, snap_py], color=style['edgecolor'],
                    linestyle='--', alpha=0.3, linewidth=1.2, zorder=1)


def draw_average_markers(ax, entry, direction_mult):
    for key, style in AVG_STYLE_MAP.items():
        loc = entry.get(key)
        if loc is None or loc[0] is None:
            continue

        avg_x, avg_y = loc
        avg_px = -direction_mult * avg_y
        avg_py = direction_mult * avg_x

        ax.scatter(avg_px, avg_py, s=400, marker='X',
                   facecolors=style['color'], edgecolors='black',
                   linewidths=1.5, alpha=0.9, zorder=5, label=style['label'])

def draw_player_control_areas(ax, entry, direction_mult, groups=None, radius=3):
    active_groups = CONTROL_AREA_GROUPS if groups is None else {k: v for k, v in CONTROL_AREA_GROUPS.items() if k in groups}

    for group_key, group_style in active_groups.items():
        points = []
        for player in entry['players']:
            if player['position_group'] in group_style['pos_groups']:
                snap_px = -direction_mult * player['y_player_ball_norm_at_snap']
                snap_py = direction_mult * player['x_player_ball_norm_at_snap']
                points.append((snap_px, snap_py))

        if not points:
            continue

        # Buffer each individual player position by `radius`, then merge overlaps
        circles = [Point(px, py).buffer(radius) for px, py in points]
        merged = unary_union(circles)

        # merged can be a single Polygon or a MultiPolygon if circles don't overlap
        geoms = merged.geoms if merged.geom_type == 'MultiPolygon' else [merged]

        for i, geom in enumerate(geoms):
            patch = MplPolygon(
                list(geom.exterior.coords),
                closed=True,
                facecolor=group_style['color'],
                edgecolor=group_style['color'],
                alpha=0.15,
                linewidth=1.0,
                linestyle='--',
                zorder=0,
                label=group_style['label'] if i == 0 else None  # avoid duplicate legend entries
            )
            ax.add_patch(patch)

def draw_control_areas(ax, entry, direction_mult):
    for group_key, group_style in CONTROL_AREA_GROUPS.items():
        points = []
        for player in entry['players']:
            if player['position_group'] in group_style['pos_groups']:
                snap_px = -direction_mult * player['y_player_ball_norm_at_snap']
                snap_py = direction_mult * player['x_player_ball_norm_at_snap']
                points.append((snap_px, snap_py))

        if len(points) < 3:
            continue

        points_arr = np.array(points)
        hull = ConvexHull(points_arr)
        hull_pts = points_arr[hull.vertices]

        buffered = ShapelyPolygon(hull_pts).buffer(2)

        patch = MplPolygon(
            list(buffered.exterior.coords),
            closed=True,
            facecolor=group_style['color'],
            edgecolor=group_style['color'],
            alpha=0.15,
            linewidth=1.0,
            linestyle='--',
            zorder=0,
            label=group_style['label']
        )
        ax.add_patch(patch)

In [ ]:
def plot_formation(results, game_id, gsis_play_id):
    entry = results.get((game_id, gsis_play_id))
    # print(entry)
    if entry is None:
        print(f"No entry found for game {game_id}, play {gsis_play_id}")
        return

    direction_mult = 1 if entry['qb_dir'] == 'positive' else -1

    backset_formation = entry.get('backfield_desc_with_motion', 'Unknown')
    if backset_formation == 'Unknown':
        print(game_id, gsis_play_id)
        return

    left_label = _side_label(entry.get('left_count'), entry.get('detached_left_count'))
    right_label = _side_label(entry.get('right_count'), entry.get('detached_right_count'))
    wr_formation = f"{left_label} - {right_label}"

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.set_facecolor('#fafafa')

    # --- Toggle these on/off by commenting out ---
    draw_players(ax, entry, direction_mult)
    # draw_average_markers(ax, entry, direction_mult)
    # draw_control_areas(ax, entry, direction_mult)
    # draw_player_control_areas(ax, entry, direction_mult)

    ax.axhline(0, color='gray', linewidth=1, linestyle='--', alpha=0.5)  # line of scrimmage

    ax.set_xlim(-25, 25)
    ax.set_ylim(-25, 25)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    ax.set_title(f"Game {game_id} -- Play {gsis_play_id} -- {backset_formation} -- {wr_formation}")
    plt.tight_layout()
    plt.show()

This per-player motion visualization loop from the original motion-classification section is superseded by the unified section below. It referenced an undefined `result_df` (never defined anywhere in this notebook) and reassigned the name `results`, which clobbered the formation-classification dictionary built earlier. See **"Unified formation + motion output"** below for the combined replacement.

## Unified formation + motion output

Combines the formation-classification results (`formation_results`) with the motion-classification logic above into:
- one visual per motioning player, showing that play's formation **and** the motioning player's classified path in the same frame
- one output row per motioning player: `game_id`, `gsis_play_id`, classified formation, classified motion

Plays with multiple motioning players produce multiple visuals/rows (one per motioning player).

In [ ]:
# Motion classification needs every frame (not just line_set), so build this
# from the unfiltered `raw_tracking_data` loaded above -- no need to read
# phi_dal_tracking_2025_1.csv a second time, and no separate column renaming
# needed since that already happened once, up top.
motion_tracking_data = raw_tracking_data.sort_values(
    ['game_id', 'gsis_play_id', 'nflId', 'frame_id']
).reset_index(drop=True)

In [ ]:
# `play_direction` isn't a column in the raw tracking file, but `classify_motions`
# needs it to know which way is "forward" for this play. Derive it per play from
# the QB's position relative to the ball: if the QB's x is greater than the ball's
# x (at snap), the offense is lined up facing toward decreasing x -> 'left';
# otherwise 'right'.
qb_rows_all = motion_tracking_data[motion_tracking_data['position_group'] == 'QB']

def _resolve_play_direction(g):
    snap_rows = g[g['event'] == 'ball_snap']
    ref = snap_rows.iloc[0] if len(snap_rows) else g.iloc[0]
    return 'left' if ref['x'] > ref['x_ball_at_snap'] else 'right'

play_direction_lookup = (
    qb_rows_all
    .groupby(['game_id', 'gsis_play_id'])
    .apply(_resolve_play_direction, include_groups=False)
    .rename('play_direction')
    .reset_index()
)

motion_tracking_data = motion_tracking_data.merge(
    play_direction_lookup, on=['game_id', 'gsis_play_id'], how='left'
)

missing_direction = motion_tracking_data['play_direction'].isna().sum()
if missing_direction:
    print(f"WARNING: {missing_direction} rows have no QB row to derive play_direction from")

In [ ]:
def plot_formation_with_motion(formation_entry, motion_info, motion_player_frames):
    """Combined visual: the play's formation at the snap (all 22 players, via the
    existing `draw_players` helper) with one motioning player's full path
    overlaid and colored by its classified motion category, plus that player's
    speed and heading-change-rate over the course of the play (same panels as
    `plot_motion`).

    formation_entry: one value from `formation_results[(game_id, gsis_play_id)]`.
    motion_info: one value from `classify_motions(...)` for a single nflId --
        has 'category', 'start_frame', 'end_frame'.
    motion_player_frames: that player's full-play rows from `motion_tracking_data`
        (all frames, any order -- this function sorts them).
    """
    game_id = formation_entry['gameId']
    gsis_play_id = formation_entry['gsisPlayId']
    direction_mult = 1 if formation_entry['qb_dir'] == 'positive' else -1

    formation_label = formation_entry.get('backfield_desc_with_motion', 'Unknown')
    category = motion_info.get('category', 'unknown')

    frames = motion_player_frames.sort_values('frame_id').reset_index(drop=True)
    snap_frame = frames.loc[frames['event'] == 'ball_snap', 'frame_id'].iloc[0]

    fig, (ax_map, ax_speed, ax_rate) = plt.subplots(1, 3, figsize=(19, 6))
    ax_map.set_facecolor('#fafafa')

    # --- Formation snapshot: reuses the existing helper, which already draws
    # every player at their snap position (dimmed pre-snap dot + line, for
    # anyone who moved). ---
    draw_players(ax_map, formation_entry, direction_mult)

    # Overlay this player's full motion path in the same ball-normalized,
    # direction-corrected coordinate space `draw_players` uses.
    x_ball = formation_entry['x_ball_at_snap']
    y_ball = formation_entry['y_ball_at_snap']

    x_norm = x_ball - frames['x']
    y_norm = y_ball - frames['y']
    path_px = -direction_mult * y_norm
    path_py = direction_mult * x_norm

    color = CATEGORY_COLORS.get(category, 'steelblue')

    start_frame, end_frame = motion_info['start_frame'], motion_info['end_frame']
    start_matches = frames.index[frames['frame_id'] == start_frame]
    end_matches = frames.index[frames['frame_id'] == end_frame]
    start_i = start_matches[0] if len(start_matches) else 0
    end_i = end_matches[0] if len(end_matches) else len(frames) - 1

    ax_map.plot(path_px.iloc[start_i:end_i + 1], path_py.iloc[start_i:end_i + 1],
                color=color, linewidth=2.5, zorder=6, label=f'Motion ({category})')
    ax_map.scatter(path_px.iloc[start_i], path_py.iloc[start_i], color='green', marker='o',
                   s=110, zorder=7, label='Motion start')
    ax_map.scatter(path_px.iloc[end_i], path_py.iloc[end_i], color='black', marker='s',
                   s=110, zorder=7, label='Motion end')

    ax_map.axhline(0, color='gray', linewidth=1, linestyle='--', alpha=0.5)  # line of scrimmage
    ax_map.set_xlim(-25, 25)
    ax_map.set_ylim(-25, 25)
    ax_map.set_aspect('equal')
    ax_map.set_xticks([])
    ax_map.set_yticks([])

    # de-duplicate legend entries from draw_players + the motion overlay
    handles, labels = ax_map.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax_map.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize=7, framealpha=0.9)

    player_name = motion_info.get('player_name', 'Unknown')
    motion_index = motion_info.get('motion_index', 0)
    ax_map.set_title(
        f"Game {game_id} -- Play {gsis_play_id} -- {formation_label}\n"
        f"Motion: {player_name} #{motion_index} ({category})"
    )

    # --- Speed panel (jet threshold + stop threshold) ---
    ax_speed.plot(frames['frame_id'], frames['s'], color='steelblue', linewidth=1.2, zorder=1)
    ax_speed.axvspan(start_frame, end_frame, color=color, alpha=0.15, zorder=0, label='Motion segment')
    ax_speed.axvline(snap_frame, color='black', linestyle='--', linewidth=1, alpha=0.7, label='Snap Frame')
    ax_speed.axhline(JET_SPEED_THRESH - 1.5, color='black', linestyle='--', linewidth=1, alpha=0.7, label='End jet speed threshold')
    ax_speed.axhline(JET_SPEED_THRESH, color='crimson', linestyle='--', linewidth=1, alpha=0.7, label='Jet speed threshold')
    ax_speed.axhline(STOP_SPEED_THRESH, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Stop threshold')
    ax_speed.set_xlim(frames['frame_id'].min(), frames['frame_id'].max())
    ax_speed.xaxis.set_major_locator(MultipleLocator(10))
    ax_speed.set_title("Speed")
    ax_speed.set_xlabel("Frame ID")
    ax_speed.set_ylabel("Yards / sec")
    ax_speed.legend(loc='best', fontsize=7)

    # --- Angle-change-rate panel (route-stabilize / return threshold) ---
    angle_change = _smoothed_angle_change(frames['dir'])
    ax_rate.plot(frames['frame_id'], angle_change, color='steelblue', linewidth=1.2, zorder=1)
    ax_rate.axvspan(start_frame, end_frame, color=color, alpha=0.15, zorder=0)
    ax_rate.axhline(ROUTE_ANGLE_THRESH, color='seagreen', linestyle='--', linewidth=1, alpha=0.7, label='Route-stabilize threshold')
    ax_rate.set_xlim(frames['frame_id'].min(), frames['frame_id'].max())
    ax_rate.xaxis.set_major_locator(MultipleLocator(10))
    ax_rate.set_title("Angle Change Rate")
    ax_rate.set_xlabel("Frame ID")
    ax_rate.set_ylabel("Degrees / frame")
    ax_rate.legend(loc='best', fontsize=7)

    plt.tight_layout()
    plt.show()

In [ ]:
combined_rows = []

unique_plays = motion_tracking_data[['game_id', 'gsis_play_id']].drop_duplicates()

for _, row in unique_plays.iterrows():
    game_id, gsis_play_id = row['game_id'], row['gsis_play_id']

    formation_entry = formation_results.get((game_id, gsis_play_id))
    if formation_entry is None:
        continue  # no formation classification available for this play

    play_motions = classify_motions(motion_tracking_data, game_id, gsis_play_id)
    if not play_motions:
        continue  # nobody in motion on this play

    play_frames = motion_tracking_data[
        (motion_tracking_data['game_id'] == game_id) &
        (motion_tracking_data['gsis_play_id'] == gsis_play_id)
    ]

    # multiple motioning players on the same play -> one visual + one row each;
    # a single player with more than one distinct motion (e.g. a return-style
    # shift followed later by a separate jet motion) also gets one visual +
    # one row per motion, keyed by (nflId, motion_index)
    for (nflId, motion_index), motion_info in play_motions.items():
        player_frames = play_frames[play_frames['nflId'] == nflId]

        # plot_formation_with_motion(formation_entry, motion_info, player_frames)

        combined_rows.append({
            'game_id': game_id,
            'gsis_play_id': gsis_play_id,
            'formation': formation_entry.get('backfield_desc_with_motion', 'Unknown'),
            'motion': motion_info['category'],
        })


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

In [ ]:
def extract_motion_features(df, game_id, gsis_play_id, nflId, start_frame, end_frame):
    """Build a fixed-length feature vector summarizing one player's motion
    segment (start_frame through end_frame, inclusive of both), for use as
    ML model input. Returns a dict, or None if the play/segment can't be
    found. `df` is expected to be motion_tracking_data (or anything with the
    same columns): game_id, gsis_play_id, nflId, frame_id, x, y, s, a, dir,
    event, x_ball_at_snap, y_ball_at_snap, play_direction, position_group,
    teamId, qbAlignment."""
    play_df = df[(df['game_id'] == game_id) & (df['gsis_play_id'] == gsis_play_id)]
    if play_df.empty:
        return None

    ball_x = play_df['x_ball_at_snap'].iloc[0]
    ball_y = play_df['y_ball_at_snap'].iloc[0]
    direction_mult = -1 if play_df['play_direction'].iloc[0] == 'left' else 1

    qb_rows = play_df[play_df['position_group'] == 'QB']
    qb_x = qb_y = None
    if not qb_rows.empty and 'ball_snap' in qb_rows['event'].values:
        qb_snap = qb_rows[qb_rows['event'] == 'ball_snap'].iloc[-1]
        qb_x, qb_y = qb_snap['x'], qb_snap['y']

    snap_rows = play_df[play_df['event'] == 'ball_snap']
    snap_frame = snap_rows['frame_id'].iloc[0] if len(snap_rows) else None

    p_df = play_df[play_df['nflId'] == nflId].sort_values('frame_id').reset_index(drop=True)
    if p_df.empty:
        return None

    start_matches = p_df.index[p_df['frame_id'] == start_frame]
    end_matches = p_df.index[p_df['frame_id'] == end_frame]
    if len(start_matches) == 0 or len(end_matches) == 0:
        return None

    seg = p_df.iloc[start_matches[0]:end_matches[0] + 1].reset_index(drop=True)
    if len(seg) < 2:
        return None

    dx = seg['x'].diff()
    dy = seg['y'].diff()
    dist_per_frame = np.sqrt(dx**2 + dy**2)
    total_distance = float(dist_per_frame.sum())

    start_x, start_y = seg['x'].iloc[0], seg['y'].iloc[0]
    end_x, end_y = seg['x'].iloc[-1], seg['y'].iloc[-1]
    net_displacement = float(np.hypot(end_x - start_x, end_y - start_y))

    dir_vals = seg['dir'].ffill().bfill().values
    turn_diffs = np.diff(dir_vals) if len(dir_vals) > 1 else np.array([0.0])
    turn_diffs = (turn_diffs + 180) % 360 - 180
    net_signed_turn = float(turn_diffs.sum())
    total_abs_turn = float(np.abs(turn_diffs).sum())

    start_y_rel_ball = start_y - ball_y
    end_y_rel_ball = end_y - ball_y
    crosses_ball_y = int(np.sign(start_y_rel_ball) != np.sign(end_y_rel_ball))

    depth_into_backfield = (seg['x'] - ball_x) * (-direction_mult)

    features = {
        'duration_frames': len(seg),
        'total_distance': total_distance,
        'net_displacement': net_displacement,
        'straightness': (net_displacement / total_distance) if total_distance > 0 else 0.0,
        'max_speed': float(seg['s'].max()),
        'avg_speed': float(seg['s'].mean()),
        'start_speed': float(seg['s'].iloc[0]),
        'end_speed': float(seg['s'].iloc[-1]),
        'max_accel': float(seg['a'].max()) if 'a' in seg.columns else np.nan,
        'net_signed_turn': net_signed_turn,
        'total_abs_turn': total_abs_turn,
        'start_x_rel_ball': (start_x - ball_x) * direction_mult,
        'start_y_rel_ball': start_y_rel_ball,
        'end_x_rel_ball': (end_x - ball_x) * direction_mult,
        'end_y_rel_ball': end_y_rel_ball,
        'crosses_ball_y': crosses_ball_y,
        'max_depth_into_backfield': float(depth_into_backfield.max()),
        'position_group': p_df['position_group'].iloc[0] if 'position_group' in p_df.columns else None,
        'team_id': p_df['teamId'].iloc[0] if 'teamId' in p_df.columns else None,
        'qb_alignment': p_df['qbAlignment'].iloc[0] if 'qbAlignment' in p_df.columns else None,
    }

    if qb_x is not None:
        features['start_x_rel_qb'] = (start_x - qb_x) * (-direction_mult)
        features['start_y_rel_qb'] = start_y - qb_y
        features['end_x_rel_qb'] = (end_x - qb_x) * (-direction_mult)
        features['end_y_rel_qb'] = end_y - qb_y
        features['start_dist_from_qb'] = float(np.hypot(start_x - qb_x, start_y - qb_y))
        features['end_dist_from_qb'] = float(np.hypot(end_x - qb_x, end_y - qb_y))
    else:
        for k in ['start_x_rel_qb', 'start_y_rel_qb', 'end_x_rel_qb', 'end_y_rel_qb',
                  'start_dist_from_qb', 'end_dist_from_qb']:
            features[k] = np.nan

    if snap_frame is not None:
        snap_matches_p = p_df.index[p_df['frame_id'] == snap_frame]
        if len(snap_matches_p):
            features['frames_to_snap_start'] = int(start_matches[0] - snap_matches_p[0])
            features['frames_to_snap_end'] = int(end_matches[0] - snap_matches_p[0])
        else:
            features['frames_to_snap_start'] = np.nan
            features['frames_to_snap_end'] = np.nan
    else:
        features['frames_to_snap_start'] = np.nan
        features['frames_to_snap_end'] = np.nan

    return features

### Step 1 -- generate the labeling template

In [ ]:
# --- Generate a labeling template: every candidate motion across whatever
# games are currently loaded in motion_tracking_data. Hand-label the
# resulting CSV's `manual_label` column (leave rule_based_category as
# reference only -- don't just copy it, that would just re-encode the same
# rule-based mistakes rather than give the model anything new to learn from).
label_rows = []

unique_plays_ml = motion_tracking_data[['game_id', 'gsis_play_id']].drop_duplicates()
for _, row in unique_plays_ml.iterrows():
    game_id, gsis_play_id = row['game_id'], row['gsis_play_id']
    play_motions = classify_motions(motion_tracking_data, game_id, gsis_play_id)
    for (nflId, motion_index), motion_info in play_motions.items():
        label_rows.append({
            'game_id': game_id,
            'gsis_play_id': gsis_play_id,
            'nflId': nflId,
            'motion_index': motion_index,
            'player_name': motion_info.get('player_name'),
            'start_frame': motion_info['start_frame'],
            'end_frame': motion_info['end_frame'],
            'rule_based_category': motion_info['category'],
        })

labeling_template = pd.DataFrame(label_rows)
labeling_template.to_csv('results_motions/motion_labels_template.csv', index=False)
print(f"Wrote {len(labeling_template)} candidate motions to motion_labeling_template.csv")
labeling_template.head()

Wrote 83 candidate motions to motion_labeling_template.csv


,game_id,gsis_play_id,nflId,motion_index,player_name,start_frame,end_frame,rule_based_category
0,2025090800,1052,46093,0,DJ Moore,2025090800-1052-079,2025090800-1052-227,Return
1,2025090800,1052,52451,0,Cole Kmet,2025090800-1052-079,2025090800-1052-227,Short
2,2025090800,1052,58212,0,Colston Loveland,2025090800-1052-079,2025090800-1052-227,Short
3,2025090800,1098,48374,0,Olamide Zaccheaus,2025090800-1098-018,2025090800-1098-111,Across
4,2025090800,1144,52451,0,Cole Kmet,2025090800-1144-039,2025090800-1144-080,Out


In [ ]:
# --- Backfield formation labeling template: one row per play ---
backfield_rows = []

for (game_id, gsis_play_id), entry in formation_results.items():
    backfield_rows.append({
        'game_id': game_id,
        'gsis_play_id': gsis_play_id,
        'qb_alignment': entry.get('qb_alignment'),
        'strength': entry.get('formation_strength'),
        'backfield_class': entry.get('backfield_desc'),  # rule engine's current prediction, for reference
    })

backfield_template = pd.DataFrame(backfield_rows)
backfield_template.to_csv('results_backset_formations/backset_labels_template.csv', index=False)
print(f"Wrote {len(backfield_template)} plays to backfield_labels_rule_based.csv")
backfield_template.head()

Wrote 117 plays to backfield_labels_rule_based.csv


,game_id,gsis_play_id,qb_alignment,strength,backfield_class
0,2025090800,1027,SHOTGUN,positive,Shotgun Empty
1,2025090800,1098,SHOTGUN,negative,Shotgun Weak
2,2025090800,1144,SHOTGUN,negative,Shotgun Strong
3,2025090800,1166,SHOTGUN,positive,Shotgun Empty
4,2025090800,1196,UNDER_CENTER,positive,Ace


In [ ]:
# --- WR/receiver-side formation labeling template: one row per play ---
# wr_class is pre-filled using the Nix/Open/Closed/Slot/... CASE logic,
# applied to each side's (count, detached_count) from formation_results,
# combined into a single "left - right" class per play.

wr_rows = []

for (game_id, gsis_play_id), entry in formation_results.items():
    left_label = _side_label(entry.get('left_count'), entry.get('detached_left_count'))
    right_label = _side_label(entry.get('right_count'), entry.get('detached_right_count'))

    wr_rows.append({
        'game_id': game_id,
        'gsis_play_id': gsis_play_id,
        'left_count': entry.get('left_count'),
        'right_count': entry.get('right_count'),
        'wr_class': f"{left_label} - {right_label}",
    })

wr_template = pd.DataFrame(wr_rows)
wr_template.to_csv('results_wr_formations/wr_labels_template.csv', index=False)
print(f"Wrote {len(wr_template)} plays to wr_labels_rule_based.csv")
wr_template.head()

Wrote 117 plays to wr_labels_rule_based.csv


,game_id,gsis_play_id,left_count,right_count,wr_class
0,2025090800,1027,2.0,3.0,Pro - Trips
1,2025090800,1098,1.0,3.0,Open - Trey
2,2025090800,1144,1.0,3.0,Open - Trey
3,2025090800,1166,2.0,3.0,Pro - Trey
4,2025090800,1196,2.0,2.0,Slot - Pro


In [ ]:
rows = []
for keys, value in formation_results.items():
    # plot_formation(formation_results, value['gameId'], value['gsisPlayId'])
    entry = {
        'game_id': value['gameId'],
        'gsis_play_id': value['gsisPlayId'],
        'team_id': value['teamId'],
        'offense_formation_raw': value['formation_raw'],
        'qb_alignment': value['qb_alignment'],
        'backfield_desc': value['backfield_desc'],
        'backfield_desc_with_motion': value['backfield_desc_with_motion'],
        'backfield_depth': value['backfield_depth'],
        'formation_strength': value['formation_strength'],
        'team_in_trips': value['team_in_trips'],
        'sniffer': value['sniffer'],
        'left_count': value['left_count'],
        'right_count': value['right_count'],
        'tight_left_count': value['tight_left_count'],
        'tight_right_count': value['tight_right_count'],
        'detached_left_count': value['detached_left_count'],
        'detached_right_count': value['detached_right_count'],
        'bunched': value['bunched'],
        'bunched_left': value['bunched_left'],
        'bunched_right': value['bunched_right'],
        'bundled': value['bundled'],
        'bundled_left': value['bundled_left'],
        'bundled_right': value['bundled_right'],
        # play outcome / coverage fields (rec_yards, rush_yards, down, play_type,
        # team_coverage_type, success, etc.) aren't in this dataset —
        # merge those in separately on (game_id, gsis_play_id) if you need them
    }
    rows.append(entry)
